# QuickBERT: Development & Experiment Notebook (Fall 2026)
> **Note:** Cells below in the *Archive* section were early prototyping drafts (superseded by the final unified `src/` implementation and real NVIDIA H100 MIG experiments starting in **Section 1**).

---
## Section 1: Environment Setup & Verification
Installs lightweight dependencies (`transformers`, `datasets`, `safetensors`, `pyyaml`) and verifies NVIDIA H100 MIG GPU, `colbert-ir/colbertv2.0` checkpoint files, and HuggingFace dataset access.

In [3]:
import os, torch, transformers, datasets, huggingface_hub
from huggingface_hub import list_repo_files

print("Working dir:", os.getcwd())
print("Existing files:", os.listdir("."))
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__)

# 1. Check ColBERTv2 checkpoint files
files = list_repo_files("colbert-ir/colbertv2.0")
print("ColBERTv2 files:", [f for f in files if f.endswith((".bin", ".safetensors", ".pt", ".json"))])

# 2. Check MS MARCO streaming access
ds = datasets.load_dataset("microsoft/ms_marco", "v2.1", split="validation", streaming=True)
row = next(iter(ds))
print("MS MARCO OK | sample query:", row["query"], "| passages:", len(row["passages"]["passage_text"]))

# 3. Check TREC DL / large-scale benchmark availability
for repo in ["irds/msmarco-passage_trec-dl-2019", "irds/msmarco-passage_trec-dl-2020", "BeIR/trec-covid", "mteb/msmarco"]:
    try:
        info = list_repo_files(repo, repo_type="dataset")
        print(f"Dataset reachable: {repo} ({len(info)} files)")
    except Exception as e:
        print(f"Dataset NOT reachable: {repo} -> {type(e).__name__}")

Working dir: /home/jovyan/QuickBERT
Existing files: ['main.tex', 'src', 'README.md', 'scripts', 'logs', 'quickbert_setup.ipynb', 'quickbert', 'results', 'QuickBERT_backup.tar.gz', '.ipynb_checkpoints', 'configs']
PyTorch: 2.10.0+cu128 | CUDA: True
GPU: NVIDIA H100 80GB HBM3 MIG 1g.10gb
transformers: 5.24.0 | datasets: 5.2.0


ColBERTv2 files: ['config.json', 'model.safetensors', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

MS MARCO OK | sample query: . what is a corporation? | passages: 10
Dataset NOT reachable: irds/msmarco-passage_trec-dl-2019 -> RepositoryNotFoundError
Dataset NOT reachable: irds/msmarco-passage_trec-dl-2020 -> RepositoryNotFoundError
Dataset reachable: BeIR/trec-covid (4 files)
Dataset reachable: mteb/msmarco (10 files)


In [4]:
import shutil
from pathlib import Path

# Remove redundant nested quickbert/ folder and old broken scripts
if Path("quickbert").exists():
    shutil.rmtree("quickbert")
for old_file in [
    "src/idf.py", "src/select.py", "src/scoring.py", "src/real_data.py",
    "src/final_results.py", "src/baseline.py", "src/run_experiments.py"
]:
    if Path(old_file).exists():
        Path(old_file).unlink()

for d in ["src", "scripts", "configs", "results/figures", "logs"]:
    Path(d).mkdir(parents=True, exist_ok=True)

Path("requirements.txt").write_text(
    "torch>=2.0.0\n"
    "transformers>=4.40.0\n"
    "datasets>=2.18.0\n"
    "huggingface_hub>=0.22.0\n"
    "safetensors>=0.4.0\n"
    "pyyaml>=6.0\n"
    "numpy>=1.24.0\n"
    "pandas>=2.0.0\n"
    "matplotlib>=3.7.0\n"
    "tqdm>=4.66.0\n"
)

Path(".gitignore").write_text(
    "__pycache__/\n"
    "*.pyc\n"
    ".ipynb_checkpoints/\n"
    "QuickBERT_backup.tar.gz\n"
    "logs/*.log\n"
    "results/*.pt\n"
    "results/*.npy\n"
    "data/\n"
    ".venv/\n"
    "venv/\n"
    ".DS_Store\n"
)

Path("configs/default.yaml").write_text(
    "# QuickBERT default configuration\n"
    "encoder: \"colbert-ir/colbertv2.0\"\n"
    "q_maxlen: 32\n"
    "d_maxlen: 180\n"
    "batch_size: 64\n"
    "k: 8\n"
    "ks: [2, 4, 8, 16, 32]\n"
    "subset_seeds: [0, 1, 2, 3, 4]\n"
    "num_eval_queries: 100\n"
    "num_passages: 1000\n"
    "large_scale_queries: 500\n"
    "large_scale_passages: 10000\n"
    "trec_passages: 5000\n"
    "timing_repeats: 5\n"
    "results_dir: \"results\"\n"
    "logs_dir: \"logs\"\n"
)

Path("src/__init__.py").write_text('"""QuickBERT: Training-Free IDF-Based Token Pruning for ColBERT."""\n')
print("Step 1A Cell 1 done: clean repo skeleton created.")

Step 1A Cell 1 done: clean repo skeleton created.


In [5]:
from pathlib import Path

# 1. src/idf_scorer.py — Author's corpus-wide IDF importance scorer
Path("src/idf_scorer.py").write_text('''"""Corpus-wide IDF importance scorer for QuickBERT.

Computes IDF(t) = log(N / df(t)) once offline over a passage collection,
mapping rare content tokens to high scores and frequent filler words to low scores.
"""
import math
from typing import Dict, Iterable, List


class IDFScorer:
    """Computes and caches corpus-level IDF weights over tokenizer vocabulary."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.df: Dict[int, int] = {}
        self.num_docs: int = 0
        self._idf_cache: Dict[int, float] = {}

    def fit(self, passages: Iterable[str]) -> "IDFScorer":
        """Compute document frequencies df(t) over a passage collection."""
        self.df = {}
        self.num_docs = 0
        for text in passages:
            self.num_docs += 1
            token_ids = self.tokenizer.encode(text, add_special_tokens=False)
            for tid in set(token_ids):
                self.df[tid] = self.df.get(tid, 0) + 1
        self._idf_cache = {
            tid: math.log(self.num_docs / freq)
            for tid, freq in self.df.items()
        }
        return self

    def idf(self, token_id: int) -> float:
        """Return IDF(t) = log(N / df(t)); unseen rare tokens receive log(N + 1)."""
        if token_id in self._idf_cache:
            return self._idf_cache[token_id]
        return math.log(self.num_docs + 1)

    def score_ids(self, token_ids: List[int]) -> List[float]:
        """Return IDF scores for a list of token IDs."""
        return [self.idf(int(tid)) for tid in token_ids]

    def inspect_query(self, query: str, k: int = 8) -> Dict[str, object]:
        """Return token-level IDF scores and kept/pruned breakdown for qualitative analysis."""
        ids = self.tokenizer.encode(query, add_special_tokens=False)
        tokens = self.tokenizer.convert_ids_to_tokens(ids)
        scores = self.score_ids(ids)
        ranked = sorted(range(len(ids)), key=lambda i: scores[i], reverse=True)
        keep_set = set(ranked[: min(k, len(ids))])
        kept = [(tokens[i], round(scores[i], 3)) for i in range(len(ids)) if i in keep_set]
        pruned = [(tokens[i], round(scores[i], 3)) for i in range(len(ids)) if i not in keep_set]
        return {"query": query, "tokens": tokens, "scores": scores, "kept": kept, "pruned": pruned}
''')

# 2. src/encoder.py — Pretrained ColBERTv2 wrapper with true 128-d projection loading
Path("src/encoder.py").write_text('''"""Pretrained ColBERTv2 token encoder with authentic 128-d linear projection."""
from typing import List, Tuple
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer


class ColBERTEncoder:
    """Encodes queries and passages into L2-normalized 128-d contextual token embeddings.

    Reuses the pretrained ColBERTv2 checkpoint (colbert-ir/colbertv2.0) without any
    fine-tuning or added parameters. Prepends ColBERT's [Q] ([unused0]) and [D] ([unused1])
    markers during encoding and strips special tokens before downstream pruning.
    """

    QUERY_MARKER_ID = 1  # [unused0] -> ColBERT [Q] marker
    DOC_MARKER_ID = 2    # [unused1] -> ColBERT [D] marker

    def __init__(
        self,
        model_name: str = "colbert-ir/colbertv2.0",
        device: str = None,
        q_maxlen: int = 32,
        d_maxlen: int = 180,
        batch_size: int = 64,
    ):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.q_maxlen = q_maxlen
        self.d_maxlen = d_maxlen
        self.batch_size = batch_size

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).eval().to(self.device)

        # Load ColBERTv2's 768 -> 128 linear projection directly from model.safetensors
        self.proj = None
        self.dim = self.model.config.hidden_size
        try:
            st_path = hf_hub_download(model_name, "model.safetensors")
            state = load_file(st_path, device="cpu")
            proj_key = next((k for k in state.keys() if k.endswith("linear.weight")), None)
            if proj_key is not None:
                weight = state[proj_key]
                self.proj = torch.nn.Linear(weight.shape[1], weight.shape[0], bias=False)
                self.proj.weight.data.copy_(weight)
                self.proj = self.proj.eval().to(self.device)
                self.dim = weight.shape[0]
                print(f"[ColBERTEncoder] Loaded {proj_key} -> dim={self.dim} on {self.device}")
            else:
                print(f"[ColBERTEncoder] No linear projection key found; using dim={self.dim}")
        except Exception as e:
            print(f"[ColBERTEncoder] Projection load fallback ({e}); using dim={self.dim}")

        self._special_ids = set(self.tokenizer.all_special_ids) | {
            self.QUERY_MARKER_ID,
            self.DOC_MARKER_ID,
        }

    @torch.no_grad()
    def _encode_batch(
        self, texts: List[str], maxlen: int, marker_id: int
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        out_embs: List[torch.Tensor] = []
        out_ids: List[torch.Tensor] = []

        use_amp = self.device == "cuda"
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start : start + self.batch_size]
            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=maxlen - 1,  # reserve 1 slot for [Q]/[D] marker
                return_tensors="pt",
            )
            input_ids = enc["input_ids"]
            attn_mask = enc["attention_mask"]

            # Insert ColBERT [Q]/[D] marker immediately after [CLS]
            bsz = input_ids.size(0)
            marker_col = torch.full((bsz, 1), marker_id, dtype=input_ids.dtype)
            ones_col = torch.ones((bsz, 1), dtype=attn_mask.dtype)
            input_ids = torch.cat([input_ids[:, :1], marker_col, input_ids[:, 1:]], dim=1).to(self.device)
            attn_mask = torch.cat([attn_mask[:, :1], ones_col, attn_mask[:, 1:]], dim=1).to(self.device)

            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                hidden = self.model(input_ids=input_ids, attention_mask=attn_mask).last_hidden_state
                if self.proj is not None:
                    hidden = self.proj(hidden)
            hidden = F.normalize(hidden.float(), p=2, dim=-1).half()

            for b in range(bsz):
                ids_b = input_ids[b]
                valid = (attn_mask[b] == 1)
                # Strip [CLS], [Q]/[D], [SEP], [PAD]
                keep = valid.clone()
                for sid in self._special_ids:
                    keep &= (ids_b != sid)
                if keep.sum().item() == 0:
                    keep[0] = True
                out_embs.append(hidden[b][keep])
                out_ids.append(ids_b[keep])

        return out_embs, out_ids

    def encode_queries(self, queries: List[str]) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """Encode queries into variable-length L2-normalized token embeddings."""
        return self._encode_batch(queries, self.q_maxlen, self.QUERY_MARKER_ID)

    def encode_docs(self, docs: List[str]) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """Encode passages into variable-length L2-normalized token embeddings."""
        return self._encode_batch(docs, self.d_maxlen, self.DOC_MARKER_ID)
''')

# 3. src/token_selector.py — Fixed-k, Dynamic-k = max(4, |Q|//2), and Random-k ablation
Path("src/token_selector.py").write_text('''"""IDF-based token selection (fixed-k, dynamic-k, and random-k ablation)."""
from typing import List, Tuple
import torch
from src.idf_scorer import IDFScorer


class TokenSelector:
    """Prunes query and document token embeddings prior to late interaction.

    Modes:
      - fixed:   keep top-k tokens by corpus IDF (default QuickBERT)
      - dynamic: per-query dynamic k = max(4, |Q| // 2), capped at max_k
      - random:  uniformly random k tokens (ablation baseline to validate IDF)
    """

    def __init__(self, idf_scorer: IDFScorer, k: int = 8, mode: str = "fixed", seed: int = 42):
        assert mode in {"fixed", "dynamic", "random"}, f"Unknown mode: {mode}"
        self.idf_scorer = idf_scorer
        self.k = k
        self.mode = mode
        self.seed = seed

    def select_batch(
        self,
        emb_list: List[torch.Tensor],
        ids_list: List[torch.Tensor],
        is_query: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, float]:
        """Select pruned token tensors and validity masks for a list of encoded texts.

        Returns:
            padded_embs: (N, max_kept, D) tensor
            valid_mask:  (N, max_kept) boolean mask
            avg_kept:    average number of real retained tokens per sequence
        """
        n = len(emb_list)
        dim = emb_list[0].shape[-1]
        device = emb_list[0].device
        dtype = emb_list[0].dtype

        # Determine per-sequence target token budget
        budgets = []
        for ids in ids_list:
            seq_len = int(ids.numel())
            if self.mode == "dynamic" and is_query:
                target = max(4, seq_len // 2)
                budgets.append(min(target, seq_len))
            else:
                budgets.append(min(self.k, seq_len))

        max_kept = max(budgets) if budgets else 1
        out = torch.zeros((n, max_kept, dim), dtype=dtype, device=device)
        mask = torch.zeros((n, max_kept), dtype=torch.bool, device=device)

        gen = torch.Generator(device="cpu").manual_seed(self.seed) if self.mode == "random" else None
        total_kept = 0

        for i in range(n):
            take = budgets[i]
            seq_len = int(ids_list[i].numel())
            total_kept += take

            if take == seq_len:
                idx = torch.arange(seq_len, device=device)
            elif self.mode == "random":
                idx = torch.randperm(seq_len, generator=gen)[:take].to(device)
            else:
                scores = torch.tensor(
                    self.idf_scorer.score_ids(ids_list[i].tolist()),
                    dtype=torch.float32,
                    device=device,
                )
                idx = torch.topk(scores, take, largest=True).indices

            out[i, :take] = emb_list[i][idx]
            mask[i, :take] = True

        return out, mask, total_kept / max(n, 1)
''')

# 4. src/sparse_maxsim.py — Exact batched GPU ColBERT MaxSim and QuickBERT Sparse MaxSim
Path("src/sparse_maxsim.py").write_text('''"""Vectorized GPU full ColBERT MaxSim and QuickBERT sparse MaxSim."""
from typing import List, Tuple
import torch


def pad_variable_embeddings(
    emb_list: List[torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor, float]:
    """Pack variable-length token embeddings into padded (N, L_max, D) tensor + mask."""
    n = len(emb_list)
    max_len = max(e.shape[0] for e in emb_list)
    dim = emb_list[0].shape[-1]
    device = emb_list[0].device
    dtype = emb_list[0].dtype

    padded = torch.zeros((n, max_len, dim), dtype=dtype, device=device)
    mask = torch.zeros((n, max_len), dtype=torch.bool, device=device)
    total_len = 0
    for i, e in enumerate(emb_list):
        length = e.shape[0]
        total_len += length
        padded[i, :length] = e
        mask[i, :length] = True
    return padded, mask, total_len / max(n, 1)


@torch.no_grad()
def full_maxsim(
    query_emb: torch.Tensor,
    doc_emb: torch.Tensor,
    doc_mask: torch.Tensor,
    chunk_size: int = 2048,
) -> torch.Tensor:
    """Compute exact full ColBERT MaxSim between one query and N candidate documents.

    Args:
        query_emb: (Lq, D) L2-normalized query token embeddings
        doc_emb:   (N, Ld, D) padded document token embeddings
        doc_mask:  (N, Ld) boolean mask indicating valid document tokens
        chunk_size: number of documents scored per GPU matmul chunk
    Returns:
        scores: (N,) relevance scores
    """
    n_docs = doc_emb.shape[0]
    q_t = query_emb.to(doc_emb.dtype).transpose(0, 1)  # (D, Lq)
    scores = torch.empty(n_docs, dtype=torch.float32, device=doc_emb.device)

    for start in range(0, n_docs, chunk_size):
        end = min(start + chunk_size, n_docs)
        # (C, Ld, D) @ (D, Lq) -> (C, Ld, Lq)
        sim = torch.matmul(doc_emb[start:end], q_t)
        sim.masked_fill_(~doc_mask[start:end].unsqueeze(-1), -1e4)
        scores[start:end] = sim.max(dim=1).values.float().sum(dim=-1)
    return scores


@torch.no_grad()
def sparse_maxsim(
    query_emb: torch.Tensor,
    query_mask: torch.Tensor,
    doc_emb: torch.Tensor,
    doc_mask: torch.Tensor,
    chunk_size: int = 4096,
) -> torch.Tensor:
    """Compute QuickBERT sparse MaxSim over top-k query and document tokens.

    Args:
        query_emb:  (Kq, D) pruned query token embeddings
        query_mask: (Kq,) boolean validity mask for query tokens
        doc_emb:    (N, Kd, D) pruned document token embeddings
        doc_mask:   (N, Kd) boolean validity mask for document tokens
        chunk_size: number of documents scored per GPU matmul chunk
    Returns:
        scores: (N,) relevance scores
    """
    n_docs = doc_emb.shape[0]
    q_t = query_emb.to(doc_emb.dtype).transpose(0, 1)  # (D, Kq)
    qm = query_mask.float().unsqueeze(0)               # (1, Kq)
    scores = torch.empty(n_docs, dtype=torch.float32, device=doc_emb.device)

    for start in range(0, n_docs, chunk_size):
        end = min(start + chunk_size, n_docs)
        # (C, Kd, D) @ (D, Kq) -> (C, Kd, Kq)
        sim = torch.matmul(doc_emb[start:end], q_t)
        sim.masked_fill_(~doc_mask[start:end].unsqueeze(-1), -1e4)
        max_sims = sim.max(dim=1).values.float()
        scores[start:end] = (max_sims * qm).sum(dim=-1)
    return scores
''')

print("Step 1A Cell 2 done: idf_scorer.py, encoder.py, token_selector.py, sparse_maxsim.py written.")

Step 1A Cell 2 done: idf_scorer.py, encoder.py, token_selector.py, sparse_maxsim.py written.


In [6]:
from pathlib import Path

# 5. src/data.py — Real MS MARCO (v2.1) and BEIR TREC-COVID data loaders
Path("src/data.py").write_text('''"""Dataset loaders for MS MARCO passage retrieval and BEIR TREC-COVID."""
from typing import Dict, List, Set, Tuple
import random
from datasets import load_dataset


_MSMARCO_CACHE = {}


def _get_msmarco_validation():
    if "val" not in _MSMARCO_CACHE:
        _MSMARCO_CACHE["val"] = load_dataset("microsoft/ms_marco", "v2.1", split="validation")
    return _MSMARCO_CACHE["val"]


def build_msmarco_subset(
    seed: int = 0,
    num_eval_queries: int = 100,
    num_passages: int = 1000,
) -> Tuple[List[str], List[str], List[Set[int]]]:
    """Build a reproducible MS MARCO mini-corpus with guaranteed gold passages.

    Guarantees:
      - Exactly `num_eval_queries` queries that have >= 1 official positive passage.
      - Exactly `num_passages` unique passages, with all gold passages included first
        followed by MS MARCO candidate hard negatives and random corpus distractors.
    """
    ds = _get_msmarco_validation().shuffle(seed=seed)

    queries: List[str] = []
    query_gold_texts: List[List[str]] = []
    hard_neg_texts: List[str] = []
    filler_texts: List[str] = []

    scan_limit = min(len(ds), max(num_eval_queries * 10, num_passages // 2 + 500))
    for i in range(scan_limit):
        row = ds[i]
        passages = row["passages"]["passage_text"]
        selected = row["passages"]["is_selected"]
        pos = [p for p, s in zip(passages, selected) if s == 1 and p.strip()]
        neg = [p for p, s in zip(passages, selected) if s == 0 and p.strip()]

        if len(queries) < num_eval_queries and pos:
            queries.append(row["query"])
            query_gold_texts.append(pos)
            hard_neg_texts.extend(neg)
        else:
            filler_texts.extend([p for p in passages if p.strip()])

        if len(queries) >= num_eval_queries and (len(hard_neg_texts) + len(filler_texts)) >= num_passages * 2:
            break

    corpus: List[str] = []
    text_to_idx: Dict[str, int] = {}

    def add_passage(txt: str) -> int:
        if txt not in text_to_idx:
            text_to_idx[txt] = len(corpus)
            corpus.append(txt)
        return text_to_idx[txt]

    # 1. Insert all positive passages first so none can ever be truncated
    gold_sets: List[Set[int]] = []
    for pos_list in query_gold_texts:
        gset = {add_passage(p) for p in pos_list}
        gold_sets.append(gset)

    # 2. Fill remaining slots with hard negatives, then random distractors
    for txt in hard_neg_texts:
        if len(corpus) >= num_passages:
            break
        add_passage(txt)

    for txt in filler_texts:
        if len(corpus) >= num_passages:
            break
        add_passage(txt)

    return queries, corpus, gold_sets


def build_trec_covid_subset(
    num_passages: int = 5000,
    seed: int = 42,
) -> Tuple[List[str], List[str], List[Dict[int, int]]]:
    """Load all 50 official TREC-COVID queries (BEIR) with graded relevance judgments.

    Includes all judged relevant passages plus biomedical distractors up to `num_passages`.
    Returns:
        queries:    list of query texts
        corpus:     list of passage texts (title + text)
        qrels_list: list of dicts mapping corpus_idx -> relevance grade (1 or 2)
    """
    q_ds = load_dataset("BeIR/trec-covid", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/trec-covid-qrels", split="test")
    corpus_ds = load_dataset("BeIR/trec-covid", "corpus", split="corpus")

    # Map query_id -> {doc_id: score} for relevant docs (score > 0)
    qid_to_rel: Dict[str, Dict[str, int]] = {}
    needed_doc_ids: Set[str] = set()
    for row in qrels_ds:
        score = int(row["score"])
        if score > 0:
            qid = str(row["query-id"])
            did = str(row["corpus-id"])
            qid_to_rel.setdefault(qid, {})[did] = score
            needed_doc_ids.add(did)

    # Cap judged docs if larger than num_passages, otherwise pad with random biomedical distractors
    rng = random.Random(seed)
    all_corpus_rows = {}
    distractors = []
    for row in corpus_ds:
        did = str(row["_id"])
        txt = (row.get("title", "") + " " + row.get("text", "")).strip()
        if not txt:
            continue
        if did in needed_doc_ids:
            all_corpus_rows[did] = txt
        elif len(distractors) < num_passages:
            distractors.append((did, txt))

    rng.shuffle(distractors)
    # If judged docs alone exceed num_passages, sample up to 60% of budget from judged relevant docs evenly per query
    selected_doc_ids: List[str] = []
    seen: Set[str] = set()
    if len(all_corpus_rows) <= num_passages:
        for did in all_corpus_rows:
            selected_doc_ids.append(did)
            seen.add(did)
    else:
        # Keep top relevant passages per query first
        for qid, rel_map in qid_to_rel.items():
            ranked_dids = sorted(rel_map.keys(), key=lambda d: rel_map[d], reverse=True)
            for did in ranked_dids[: max(10, num_passages // (2 * len(qid_to_rel)))]:
                if did in all_corpus_rows and did not in seen:
                    selected_doc_ids.append(did)
                    seen.add(did)
        for did in all_corpus_rows:
            if len(selected_doc_ids) >= int(num_passages * 0.65):
                break
            if did not in seen:
                selected_doc_ids.append(did)
                seen.add(did)

    corpus: List[str] = []
    did_to_idx: Dict[str, int] = {}
    for did in selected_doc_ids:
        did_to_idx[did] = len(corpus)
        corpus.append(all_corpus_rows[did])

    for did, txt in distractors:
        if len(corpus) >= num_passages:
            break
        if did not in did_to_idx:
            did_to_idx[did] = len(corpus)
            corpus.append(txt)

    queries: List[str] = []
    qrels_list: List[Dict[int, int]] = []
    for row in q_ds:
        qid = str(row["_id"])
        rel_map = qid_to_rel.get(qid, {})
        mapped = {did_to_idx[d]: s for d, s in rel_map.items() if d in did_to_idx}
        if mapped:
            queries.append(row["text"])
            qrels_list.append(mapped)

    return queries, corpus, qrels_list
''')

# 6. src/evaluate.py — Recall@10, MRR@10, NDCG@10, and CUDA-synchronized timing
Path("src/evaluate.py").write_text('''"""Retrieval quality metrics (Recall@10, MRR@10, NDCG@10) and GPU timing utilities."""
import math
import time
from typing import Callable, Dict, Iterable, Union
import torch


def compute_metrics(
    scores: torch.Tensor,
    gold: Union[set, Dict[int, int]],
    k: int = 10,
) -> Dict[str, float]:
    """Compute Recall@k (Hit@k), MRR@k, and NDCG@k for one query.

    Args:
        scores: (N,) relevance scores over N passages
        gold:   set of relevant indices (binary) or dict {idx: grade} (graded)
        k:      cutoff (default 10)
    """
    topk = torch.topk(scores, min(k, scores.numel())).indices.tolist()
    if isinstance(gold, dict):
        rel_map = gold
    else:
        rel_map = {idx: 1 for idx in gold}

    hit = 1.0 if any(idx in rel_map for idx in topk) else 0.0

    mrr = 0.0
    for rank, idx in enumerate(topk, start=1):
        if idx in rel_map:
            mrr = 1.0 / rank
            break

    dcg = 0.0
    for rank, idx in enumerate(topk, start=1):
        rel = rel_map.get(idx, 0)
        if rel > 0:
            dcg += (2**rel - 1) / math.log2(rank + 1)

    ideal_rels = sorted(rel_map.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(rank + 1) for rank, rel in enumerate(ideal_rels, start=1))
    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    return {"recall@10": hit, "mrr@10": mrr, "ndcg@10": ndcg}


@torch.no_grad()
def benchmark_fn(fn: Callable, repeats: int = 5, warmup: int = 2) -> float:
    """Measure average wall-clock execution time (ms) with CUDA synchronization."""
    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times_ms = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    return sum(times_ms) / len(times_ms)
''')

# 7. src/utils.py — Reproducibility, hardware metadata, and logging
Path("src/utils.py").write_text('''"""Reproducibility seeding, hardware reporting, and logging utilities."""
import json
import logging
import os
import platform
import random
import sys
import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_hardware_info() -> dict:
    return {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "cuda_version": torch.version.cuda,
        "torch_version": torch.__version__,
        "python_version": platform.python_version(),
        "cpu_count": os.cpu_count(),
    }


def get_logger(name: str = "quickbert", log_file: str = "logs/quickbert.log") -> logging.Logger:
    os.makedirs(os.path.dirname(log_file), exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(sh)
        fh = logging.FileHandler(log_file)
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger


def save_json(obj: dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
''')

print("Step 1A Cell 3 done: data.py, evaluate.py, utils.py written.")

Step 1A Cell 3 done: data.py, evaluate.py, utils.py written.


---
## Section 2: Unified QuickBERT Implementation (`src/` & `configs/`)
Builds the final training-free codebase: offline corpus `IDFScorer`, authentic 128-d `ColBERTEncoder`, `TokenSelector` (IDF, TF-IDF, dynamic-$k$, random-$k$), chunked GPU `sparse_maxsim`, MS MARCO / BEIR TREC-COVID loaders, and CUDA-synchronized evaluation utilities.

In [7]:
from src.encoder import ColBERTEncoder
from src.idf_scorer import IDFScorer
from src.token_selector import TokenSelector
from src.sparse_maxsim import pad_variable_embeddings, full_maxsim, sparse_maxsim
from src.data import build_msmarco_subset, build_trec_covid_subset
from src.evaluate import compute_metrics

# 1. Verify authentic 128-d ColBERTv2 encoder
enc = ColBERTEncoder()
assert enc.dim == 128, f"Expected 128-d ColBERTv2 embeddings, got {enc.dim}"

# 2. Verify MS MARCO loader (tiny check: 5 queries x 50 passages)
queries, corpus, golds = build_msmarco_subset(seed=0, num_eval_queries=5, num_passages=50)
print(f"MS MARCO smoke check: {len(queries)} queries, {len(corpus)} passages")

idf = IDFScorer(enc.tokenizer).fit(corpus)
print("Qualitative IDF check:", idf.inspect_query("what is the population of raleigh", k=2))

q_emb, q_ids = enc.encode_queries(queries)
d_emb, d_ids = enc.encode_docs(corpus)
D_full, M_full, avg_d = pad_variable_embeddings(d_emb)

selector = TokenSelector(idf, k=8, mode="fixed")
Q_sel, Q_mask, avg_qk = selector.select_batch(q_emb, q_ids, is_query=True)
D_sel, D_mask, avg_dk = selector.select_batch(d_emb, d_ids, is_query=False)

m_base = compute_metrics(full_maxsim(q_emb[0], D_full, M_full), golds[0])
m_qb = compute_metrics(sparse_maxsim(Q_sel[0], Q_mask[0], D_sel, D_mask), golds[0])
print(f"Sample query metrics -> Baseline: {m_base} | QuickBERT(k=8): {m_qb}")

# 3. Verify BEIR TREC-COVID access
tq, tc, tg = build_trec_covid_subset(num_passages=200)
print(f"TREC-COVID smoke check: {len(tq)} queries, {len(tc)} passages, avg rels/q={sum(len(x) for x in tg)/len(tg):.1f}")
print("ALL SMOKE TESTS PASSED!")

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (30 outputs left out)

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  111MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/171332 [00:00<?, ? examples/s]

TREC-COVID smoke check: 50 queries, 431 passages, avg rels/q=15.3
ALL SMOKE TESTS PASSED!


In [8]:
from pathlib import Path
import os

Path("scripts/run_experiments.py").write_text('''"""Unified experiment runner for QuickBERT (MS MARCO, TREC-COVID, and H100 benchmarks).

Usage examples:
  python scripts/run_experiments.py --mode baseline
  python scripts/run_experiments.py --mode quickbert --k 8
  python scripts/run_experiments.py --mode msmarco_5sub
  python scripts/run_experiments.py --mode large_scale
  python scripts/run_experiments.py --mode trec_covid
  python scripts/run_experiments.py --mode hardware_bench
  python scripts/run_experiments.py --mode all
"""
import argparse
import json
import os
import sys
import time
import yaml
import pandas as pd
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.data import build_msmarco_subset, build_trec_covid_subset
from src.encoder import ColBERTEncoder
from src.evaluate import benchmark_fn, compute_metrics
from src.idf_scorer import IDFScorer
from src.sparse_maxsim import full_maxsim, pad_variable_embeddings, sparse_maxsim
from src.token_selector import TokenSelector
from src.utils import get_hardware_info, get_logger, save_json, set_seed


def evaluate_config(
    q_embs,
    q_ids,
    d_embs,
    d_ids,
    golds,
    idf_scorer,
    mode: str,
    k: int = 8,
    repeats: int = 5,
    seed: int = 42,
):
    """Evaluate baseline or QuickBERT configuration on pre-encoded query/doc embeddings."""
    n_queries = len(q_embs)
    if mode == "baseline":
        D_mat, D_mask, avg_d = pad_variable_embeddings(d_embs)
        avg_q = sum(e.shape[0] for e in q_embs) / n_queries

        def score_all():
            return [full_maxsim(q_embs[i], D_mat, D_mask) for i in range(n_queries)]

        all_scores = score_all()
        total_ms = benchmark_fn(score_all, repeats=repeats, warmup=2)
    else:
        selector = TokenSelector(idf_scorer, k=k, mode=mode, seed=seed)
        Q_sel, Q_mask, avg_q = selector.select_batch(q_embs, q_ids, is_query=True)
        D_sel, D_mask, avg_d = selector.select_batch(d_embs, d_ids, is_query=False)

        def score_all():
            return [sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask) for i in range(n_queries)]

        all_scores = score_all()
        total_ms = benchmark_fn(score_all, repeats=repeats, warmup=2)

    recalls, mrrs, ndcgs = [], [], []
    for i, s in enumerate(all_scores):
        m = compute_metrics(s, golds[i], k=10)
        recalls.append(m["recall@10"])
        mrrs.append(m["mrr@10"])
        ndcgs.append(m["ndcg@10"])

    return {
        "recall@10": sum(recalls) / n_queries,
        "mrr@10": sum(mrrs) / n_queries,
        "ndcg@10": sum(ndcgs) / n_queries,
        "ms_per_query": total_ms / n_queries,
        "avg_q_tokens": avg_q,
        "avg_d_tokens": avg_d,
        "avg_interactions": avg_q * avg_d,
    }


def run_msmarco_5sub(encoder: ColBERTEncoder, cfg: dict, log):
    """Experiment 1: 5 random MS MARCO subsets (100 queries x 1,000 passages)."""
    log.info("=== Experiment 1: MS MARCO 5-Subset Sweep ===")
    seeds = cfg["subset_seeds"]
    ks = cfg["ks"]
    repeats = cfg["timing_repeats"]

    rows = []
    qualitative_saved = False

    for seed in seeds:
        queries, corpus, golds = build_msmarco_subset(
            seed=seed,
            num_eval_queries=cfg["num_eval_queries"],
            num_passages=cfg["num_passages"],
        )
        idf = IDFScorer(encoder.tokenizer).fit(corpus)
        q_embs, q_ids = encoder.encode_queries(queries)
        d_embs, d_ids = encoder.encode_docs(corpus)

        if not qualitative_saved:
            samples = [
                idf.inspect_query("what is the population of raleigh", k=2),
                idf.inspect_query("what is the weather in raleigh today", k=4),
                idf.inspect_query(queries[0], k=8),
                idf.inspect_query(queries[1], k=8),
            ]
            save_json({"samples": samples}, "results/qualitative_idf.json")
            qualitative_saved = True

        base = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="baseline", repeats=repeats)
        log.info(
            f"[Seed {seed}] Baseline | R@10={base['recall@10']*100:.2f}% | "
            f"MRR@10={base['mrr@10']*100:.2f}% | {base['ms_per_query']:.3f} ms/q | "
            f"pairs={base['avg_interactions']:.1f}"
        )
        rows.append({
            "seed": seed, "method": "baseline", "k": 0,
            "recall@10": base["recall@10"], "mrr@10": base["mrr@10"], "ndcg@10": base["ndcg@10"],
            "recall_drop": 0.0, "ms_per_query": base["ms_per_query"], "speedup": 1.0,
            "avg_q_tokens": base["avg_q_tokens"], "avg_d_tokens": base["avg_d_tokens"],
            "avg_interactions": base["avg_interactions"],
        })

        # Fixed-k sweep
        for k in ks:
            res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="fixed", k=k, repeats=repeats)
            spd = base["ms_per_query"] / res["ms_per_query"]
            rdrop = base["recall@10"] - res["recall@10"]
            rows.append({
                "seed": seed, "method": f"fixed_k={k}", "k": k,
                "recall@10": res["recall@10"], "mrr@10": res["mrr@10"], "ndcg@10": res["ndcg@10"],
                "recall_drop": rdrop, "ms_per_query": res["ms_per_query"], "speedup": spd,
                "avg_q_tokens": res["avg_q_tokens"], "avg_d_tokens": res["avg_d_tokens"],
                "avg_interactions": res["avg_interactions"],
            })

        # Dynamic-k ablation: k_q = max(4, |Q|//2), k_d = 8
        dyn = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="dynamic", k=8, repeats=repeats)
        rows.append({
            "seed": seed, "method": "dynamic_k", "k": 8,
            "recall@10": dyn["recall@10"], "mrr@10": dyn["mrr@10"], "ndcg@10": dyn["ndcg@10"],
            "recall_drop": base["recall@10"] - dyn["recall@10"],
            "ms_per_query": dyn["ms_per_query"], "speedup": base["ms_per_query"] / dyn["ms_per_query"],
            "avg_q_tokens": dyn["avg_q_tokens"], "avg_d_tokens": dyn["avg_d_tokens"],
            "avg_interactions": dyn["avg_interactions"],
        })

        # Random-k=8 ablation (validates IDF importance vs uniform pruning)
        rnd = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="random", k=8, repeats=repeats, seed=seed)
        rows.append({
            "seed": seed, "method": "random_k=8", "k": 8,
            "recall@10": rnd["recall@10"], "mrr@10": rnd["mrr@10"], "ndcg@10": rnd["ndcg@10"],
            "recall_drop": base["recall@10"] - rnd["recall@10"],
            "ms_per_query": rnd["ms_per_query"], "speedup": base["ms_per_query"] / rnd["ms_per_query"],
            "avg_q_tokens": rnd["avg_q_tokens"], "avg_d_tokens": rnd["avg_d_tokens"],
            "avg_interactions": rnd["avg_interactions"],
        })

    df = pd.DataFrame(rows)
    df.to_csv("results/msmarco_5sub_raw.csv", index=False)

    summary = (
        df.groupby(["method", "k"], sort=False)
        .agg(
            recall_mean=("recall@10", "mean"),
            recall_std=("recall@10", "std"),
            recall_drop_mean=("recall_drop", "mean"),
            recall_drop_std=("recall_drop", "std"),
            mrr_mean=("mrr@10", "mean"),
            ndcg_mean=("ndcg@10", "mean"),
            ms_mean=("ms_per_query", "mean"),
            speedup_mean=("speedup", "mean"),
            speedup_std=("speedup", "std"),
            avg_interactions=("avg_interactions", "mean"),
        )
        .reset_index()
    )
    summary.to_csv("results/msmarco_5sub_summary.csv", index=False)
    log.info("\n" + summary.to_string(index=False))
    return summary


def run_large_scale(encoder: ColBERTEncoder, cfg: dict, log):
    """Experiment 2A: Large-scale MS MARCO evaluation (500 queries x 10,000 passages)."""
    n_q = cfg["large_scale_queries"]
    n_p = cfg["large_scale_passages"]
    log.info(f"=== Experiment 2A: Large-Scale MS MARCO ({n_q} queries x {n_p} passages) ===")

    queries, corpus, golds = build_msmarco_subset(seed=42, num_eval_queries=n_q, num_passages=n_p)
    idf = IDFScorer(encoder.tokenizer).fit(corpus)
    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs, d_ids = encoder.encode_docs(corpus)

    base = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="baseline", repeats=3)
    rows = [{
        "method": "baseline", "k": 0, **base,
        "recall_drop": 0.0, "speedup": 1.0,
    }]
    for k in cfg["ks"]:
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode="fixed", k=k, repeats=3)
        rows.append({
            "method": f"fixed_k={k}", "k": k, **res,
            "recall_drop": base["recall@10"] - res["recall@10"],
            "speedup": base["ms_per_query"] / res["ms_per_query"],
        })
    for mode_name in ["dynamic", "random"]:
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode=mode_name, k=8, repeats=3)
        rows.append({
            "method": f"{mode_name}_k=8", "k": 8, **res,
            "recall_drop": base["recall@10"] - res["recall@10"],
            "speedup": base["ms_per_query"] / res["ms_per_query"],
        })

    df = pd.DataFrame(rows)
    df.to_csv("results/msmarco_large_scale.csv", index=False)
    log.info("\n" + df[["method", "recall@10", "mrr@10", "ndcg@10", "ms_per_query", "speedup", "avg_interactions"]].to_string(index=False))
    return df, idf


def run_trec_covid(encoder: ColBERTEncoder, cfg: dict, log, msmarco_idf=None):
    """Experiment 2B: Out-of-domain BEIR TREC-COVID evaluation (in-domain vs cross-domain IDF)."""
    n_p = cfg["trec_passages"]
    log.info(f"=== Experiment 2B: BEIR TREC-COVID ({n_p} passages, in-domain & cross-domain IDF) ===")

    queries, corpus, qrels = build_trec_covid_subset(num_passages=n_p, seed=42)
    log.info(f"Loaded TREC-COVID: {len(queries)} official queries x {len(corpus)} biomedical passages")

    in_domain_idf = IDFScorer(encoder.tokenizer).fit(corpus)
    if msmarco_idf is None:
        _, ms_corpus, _ = build_msmarco_subset(seed=42, num_eval_queries=100, num_passages=5000)
        msmarco_idf = IDFScorer(encoder.tokenizer).fit(ms_corpus)

    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs, d_ids = encoder.encode_docs(corpus)

    base = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode="baseline", repeats=5)
    rows = [{
        "idf_source": "none", "method": "baseline", "k": 0, **base,
        "ndcg_drop": 0.0, "speedup": 1.0,
    }]

    # In-domain TREC-COVID IDF sweep
    for k in cfg["ks"]:
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode="fixed", k=k, repeats=5)
        rows.append({
            "idf_source": "in_domain", "method": f"fixed_k={k}", "k": k, **res,
            "ndcg_drop": base["ndcg@10"] - res["ndcg@10"],
            "speedup": base["ms_per_query"] / res["ms_per_query"],
        })

    # Cross-domain MS MARCO IDF transfer at k=8 and k=16 (tests domain shift directly!)
    for k in [8, 16]:
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, msmarco_idf, mode="fixed", k=k, repeats=5)
        rows.append({
            "idf_source": "cross_domain_msmarco", "method": f"fixed_k={k}", "k": k, **res,
            "ndcg_drop": base["ndcg@10"] - res["ndcg@10"],
            "speedup": base["ms_per_query"] / res["ms_per_query"],
        })

    # Random-k=8 baseline on TREC-COVID
    rnd = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode="random", k=8, repeats=5)
    rows.append({
        "idf_source": "random", "method": "random_k=8", "k": 8, **rnd,
        "ndcg_drop": base["ndcg@10"] - rnd["ndcg@10"],
        "speedup": base["ms_per_query"] / rnd["ms_per_query"],
    })

    df = pd.DataFrame(rows)
    df.to_csv("results/trec_covid_results.csv", index=False)
    log.info("\n" + df[["idf_source", "method", "recall@10", "ndcg@10", "mrr@10", "speedup", "avg_interactions"]].to_string(index=False))
    return df


def run_hardware_bench(encoder: ColBERTEncoder, cfg: dict, log):
    """Experiment 3: Fixed-hardware H100 MIG scaling & end-to-end latency breakdown."""
    log.info("=== Experiment 3: Fixed-Hardware H100 MIG Latency Benchmark ===")
    hw = get_hardware_info()
    save_json(hw, "results/hardware_env.json")

    queries, corpus_10k, _ = build_msmarco_subset(seed=99, num_eval_queries=100, num_passages=10000)
    idf = IDFScorer(encoder.tokenizer).fit(corpus_10k)
    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs_all, d_ids_all = encoder.encode_docs(corpus_10k)

    # Measure online single-query encoding + IDF lookup overhead
    def encode_one_batch():
        encoder.encode_queries(queries)

    enc_ms_per_q = benchmark_fn(encode_one_batch, repeats=5, warmup=2) / len(queries)

    def prune_queries_only():
        TokenSelector(idf, k=8, mode="fixed").select_batch(q_embs, q_ids, is_query=True)

    prune_ms_per_q = benchmark_fn(prune_queries_only, repeats=5, warmup=2) / len(queries)

    rows = []
    for corpus_n in [1000, 5000, 10000]:
        d_embs = d_embs_all[:corpus_n]
        d_ids = d_ids_all[:corpus_n]
        D_full, M_full, _ = pad_variable_embeddings(d_embs)

        def base_score():
            for i in range(len(q_embs)):
                full_maxsim(q_embs[i], D_full, M_full)

        base_ms = benchmark_fn(base_score, repeats=5, warmup=2) / len(q_embs)
        rows.append({
            "corpus_size": corpus_n, "k": 0, "method": "baseline",
            "score_ms_per_q": base_ms, "encode_ms_per_q": enc_ms_per_q,
            "prune_ms_per_q": 0.0, "e2e_ms_per_q": base_ms + enc_ms_per_q,
            "score_speedup": 1.0, "e2e_speedup": 1.0,
        })

        for k in cfg["ks"]:
            sel = TokenSelector(idf, k=k, mode="fixed")
            Q_sel, Q_mask, _ = sel.select_batch(q_embs, q_ids, is_query=True)
            D_sel, D_mask, _ = sel.select_batch(d_embs, d_ids, is_query=False)

            def qb_score():
                for i in range(len(q_embs)):
                    sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask)

            qb_ms = benchmark_fn(qb_score, repeats=5, warmup=2) / len(q_embs)
            e2e_ms = qb_ms + enc_ms_per_q + prune_ms_per_q
            rows.append({
                "corpus_size": corpus_n, "k": k, "method": f"quickbert_k={k}",
                "score_ms_per_q": qb_ms, "encode_ms_per_q": enc_ms_per_q,
                "prune_ms_per_q": prune_ms_per_q, "e2e_ms_per_q": e2e_ms,
                "score_speedup": base_ms / qb_ms,
                "e2e_speedup": (base_ms + enc_ms_per_q) / e2e_ms,
            })

    df = pd.DataFrame(rows)
    df.to_csv("results/hardware_benchmark.csv", index=False)
    log.info(f"Hardware: {hw['gpu_name']} | Query encoding: {enc_ms_per_q:.3f} ms/q | Query IDF pruning: {prune_ms_per_q:.3f} ms/q")
    log.info("\n" + df.to_string(index=False))
    return df


def main():
    parser = argparse.ArgumentParser(description="Run QuickBERT retrieval & latency experiments.")
    parser.add_argument(
        "--mode",
        choices=["baseline", "quickbert", "msmarco_5sub", "large_scale", "trec_covid", "hardware_bench", "all"],
        default="all",
    )
    parser.add_argument("--config", type=str, default="configs/default.yaml")
    parser.add_argument("--k", type=int, default=None, help="Override top-k for --mode quickbert")
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    set_seed(42)
    log = get_logger()
    hw = get_hardware_info()
    log.info(f"Environment: {json.dumps(hw)}")

    encoder = ColBERTEncoder(
        model_name=cfg["encoder"],
        q_maxlen=cfg["q_maxlen"],
        d_maxlen=cfg["d_maxlen"],
        batch_size=cfg["batch_size"],
    )

    if args.mode in {"baseline", "quickbert"}:
        queries, corpus, golds = build_msmarco_subset(
            seed=0, num_eval_queries=cfg["num_eval_queries"], num_passages=cfg["num_passages"]
        )
        idf = IDFScorer(encoder.tokenizer).fit(corpus)
        q_embs, q_ids = encoder.encode_queries(queries)
        d_embs, d_ids = encoder.encode_docs(corpus)
        k_val = args.k if args.k is not None else cfg["k"]
        run_mode = "baseline" if args.mode == "baseline" else "fixed"
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, idf, mode=run_mode, k=k_val, repeats=cfg["timing_repeats"])
        log.info(f"[{args.mode.upper()} k={k_val if run_mode!='baseline' else 'full'}] {res}")
        return

    msmarco_idf = None
    if args.mode in {"msmarco_5sub", "all"}:
        run_msmarco_5sub(encoder, cfg, log)
    if args.mode in {"large_scale", "all"}:
        _, msmarco_idf = run_large_scale(encoder, cfg, log)
    if args.mode in {"trec_covid", "all"}:
        run_trec_covid(encoder, cfg, log, msmarco_idf=msmarco_idf)
    if args.mode in {"hardware_bench", "all"}:
        run_hardware_bench(encoder, cfg, log)


if __name__ == "__main__":
    main()
''')

# Clean, bug-free shell wrappers that match README & NeurIPS report snippets
Path("scripts/run_baseline.sh").write_text(
    "#!/usr/bin/env bash\nset -e\npython scripts/run_experiments.py --mode baseline --config configs/default.yaml\n"
)
Path("scripts/run_quickbert.sh").write_text(
    "#!/usr/bin/env bash\nset -e\npython scripts/run_experiments.py --mode quickbert --config configs/default.yaml \"$@\"\n"
)
Path("scripts/run_ablation.sh").write_text(
    "#!/usr/bin/env bash\nset -e\npython scripts/run_experiments.py --mode all --config configs/default.yaml\n"
)

for sh in ["scripts/run_baseline.sh", "scripts/run_quickbert.sh", "scripts/run_ablation.sh"]:
    os.chmod(sh, 0o755)

print("Step 1B Cell 1 done: run_experiments.py and shell scripts written.")

Step 1B Cell 1 done: run_experiments.py and shell scripts written.


In [11]:
!bash scripts/run_ablation.sh

2026-11-24 05:30:27,266 | INFO | Environment: {"device": "cuda", "gpu_name": "NVIDIA H100 80GB HBM3 MIG 1g.10gb", "cuda_version": "12.8", "torch_version": "2.10.0+cu128", "python_version": "3.13.11", "cpu_count": 20}
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 3775.85it/s]
[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[ColBERTEncoder] Loaded linear.weight -> dim=128 on cuda
2026-11-24 05:30:29,382 | INFO | === Experiment 1: MS MARCO 5-Subset Sweep ===
2026-11-24 05:30:32,447 | INFO | [Seed 0] Baseline | R@10=99.00% | MRR@10=60.60% | 0.306 ms/q | pairs=457.7
2026-11-24 05:30:35,305 | INFO | [Seed 1] Baseline | R@10=99.00% | MRR@10=68.21% | 0.313 ms/q | pairs=493.5
2026-11-24 05:30:38,110 | INFO | [Seed 2] Baseline | R@10=99.00%

In [12]:
import math, re
import torch
from src.data import build_msmarco_subset
from src.encoder import ColBERTEncoder
from src.idf_scorer import IDFScorer
from src.sparse_maxsim import pad_variable_embeddings, full_maxsim, sparse_maxsim
from src.evaluate import compute_metrics, benchmark_fn

enc = ColBERTEncoder()
queries, corpus, golds = build_msmarco_subset(seed=0, num_eval_queries=100, num_passages=1000)
idf = IDFScorer(enc.tokenizer).fit(corpus)
q_embs, q_ids = enc.encode_queries(queries)
d_embs, d_ids = enc.encode_docs(corpus)

# 1. Inspect what naive IDF keeps on the first gold passage!
g_idx = next(iter(golds[0]))
toks = enc.tokenizer.convert_ids_to_tokens(d_ids[g_idx].tolist())
scores = idf.score_ids(d_ids[g_idx].tolist())
top8 = sorted(range(len(toks)), key=lambda i: scores[i], reverse=True)[:8]
bot8 = sorted(range(len(toks)), key=lambda i: scores[i])[:8]
print("Query 0:", queries[0])
print("Gold passage length (tokens):", len(toks))
print("Naive IDF TOP-8 kept:", [(toks[i], round(scores[i], 2)) for i in top8])
print("Naive IDF BOT-8 dropped:", [(toks[i], round(scores[i], 2)) for i in bot8])
print("-" * 75)

# Helper to test any scoring/pruning rule on (q_embs, d_embs)
def eval_rule(name, q_rule_fn, d_rule_fn):
    def apply_rule(emb_list, ids_list, rule_fn):
        kept_embs = [emb_list[i][rule_fn(ids_list[i], emb_list[i])] for i in range(len(emb_list))]
        return pad_variable_embeddings(kept_embs)

    Q_sel, Q_mask, avg_q = apply_rule(q_embs, q_ids, q_rule_fn)
    D_sel, D_mask, avg_d = apply_rule(d_embs, d_ids, d_rule_fn)

    def run():
        return [sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask) for i in range(len(q_embs))]

    all_s = run()
    ms = benchmark_fn(run, repeats=3, warmup=1) / len(q_embs)
    r10 = sum(compute_metrics(s, golds[i])["recall@10"] for i, s in enumerate(all_s)) / len(q_embs)
    mrr = sum(compute_metrics(s, golds[i])["mrr@10"] for i, s in enumerate(all_s)) / len(q_embs)
    print(f"{name:<36} | R@10={r10*100:5.1f}% | MRR@10={mrr*100:5.1f}% | avg_q={avg_q:4.1f} | avg_d={avg_d:5.1f} | pairs={avg_q*avg_d:6.1f} | {ms:.3f} ms")

# Baseline
D_full, M_full, avg_d_full = pad_variable_embeddings(d_embs)
avg_q_full = sum(e.shape[0] for e in q_embs) / len(q_embs)
base_s = [full_maxsim(q_embs[i], D_full, M_full) for i in range(len(q_embs))]
base_ms = benchmark_fn(lambda: [full_maxsim(q_embs[i], D_full, M_full) for i in range(len(q_embs))], repeats=3, warmup=1) / len(q_embs)
base_r10 = sum(compute_metrics(s, golds[i])["recall@10"] for i, s in enumerate(base_s)) / len(q_embs)
base_mrr = sum(compute_metrics(s, golds[i])["mrr@10"] for i, s in enumerate(base_s)) / len(q_embs)
print(f"{'Baseline (Full ColBERTv2)':<36} | R@10={base_r10*100:5.1f}% | MRR@10={base_mrr*100:5.1f}% | avg_q={avg_q_full:4.1f} | avg_d={avg_d_full:5.1f} | pairs={avg_q_full*avg_d_full:6.1f} | {base_ms:.3f} ms")

# Build punctuation/subword filter helper
vocab_is_alpha = {}
def is_clean_word(tid):
    if tid not in vocab_is_alpha:
        tok = enc.tokenizer.convert_ids_to_tokens(int(tid))
        vocab_is_alpha[tid] = bool(re.match(r"^[a-zA-Z]{2,}$", tok))
    return vocab_is_alpha[tid]

def topk_idf(k, dedup=False, clean_only=False):
    def fn(ids, emb):
        id_list = ids.tolist()
        vals = []
        seen = set()
        for pos, tid in enumerate(id_list):
            s = idf.idf(tid)
            if clean_only and not is_clean_word(tid):
                s -= 10.0
            if dedup and tid in seen:
                s -= 10.0
            seen.add(tid)
            vals.append(s)
        take = min(k, len(id_list))
        return torch.topk(torch.tensor(vals, device=ids.device), take).indices
    return fn

def topk_tfidf(k, dedup=False):
    def fn(ids, emb):
        id_list = ids.tolist()
        counts = {}
        for tid in id_list:
            counts[tid] = counts.get(tid, 0) + 1
        vals = []
        seen = set()
        for tid in id_list:
            s = (1.0 + math.log(counts[tid])) * idf.idf(tid)
            if not is_clean_word(tid):
                s *= 0.25
            if dedup and tid in seen:
                s = -1.0
            seen.add(tid)
            vals.append(s)
        take = min(k, len(id_list))
        return torch.topk(torch.tensor(vals, device=ids.device), take).indices
    return fn

def topk_by_norm_x_idf(k):
    # Pre-norm or embedding salience isn't available after L2-norm, so test IDF-threshold keep-top-ratio
    def fn(ids, emb):
        id_list = ids.tolist()
        vals = [idf.idf(tid) if is_clean_word(tid) else idf.idf(tid) * 0.25 for tid in id_list]
        take = min(k, len(id_list))
        return torch.topk(torch.tensor(vals, device=ids.device), take).indices
    return fn

def keep_all(ids, emb):
    return torch.arange(ids.numel(), device=ids.device)

def keep_idf_above(thresh):
    def fn(ids, emb):
        id_list = ids.tolist()
        idx = [i for i, tid in enumerate(id_list) if idf.idf(tid) >= thresh and is_clean_word(tid)]
        if len(idx) < 4:
            scores = torch.tensor(idf.score_ids(id_list), device=ids.device)
            return torch.topk(scores, min(4, len(id_list))).indices
        return torch.tensor(idx, device=ids.device)
    return fn

# Run diagnostics
eval_rule("Naive IDF (k_q=8, k_d=8)", topk_idf(8), topk_idf(8))
eval_rule("Clean+Dedup IDF (k_q=8, k_d=8)", topk_idf(8, True, True), topk_idf(8, True, True))
eval_rule("Clean+Dedup TF-IDF (k_q=8, k_d=8)", topk_tfidf(8, True), topk_tfidf(8, True))
eval_rule("Clean+Dedup TF-IDF (k_q=8, k_d=16)", topk_tfidf(8, True), topk_tfidf(16, True))
eval_rule("Clean+Dedup TF-IDF (k_q=8, k_d=24)", topk_tfidf(8, True), topk_tfidf(24, True))
eval_rule("Clean+Dedup TF-IDF (k_q=8, k_d=32)", topk_tfidf(8, True), topk_tfidf(32, True))
eval_rule("Clean IDF-Filter >= 2.0 (Q & D)", keep_idf_above(2.0), keep_idf_above(2.0))
eval_rule("Clean IDF-Filter >= 2.5 (Q & D)", keep_idf_above(2.5), keep_idf_above(2.5))
eval_rule("Clean IDF-Filter >= 3.0 (Q & D)", keep_idf_above(3.0), keep_idf_above(3.0))
eval_rule("Clean IDF (k_q=6, k_d=24)", topk_idf(6, True, True), topk_idf(24, True, True))
eval_rule("Clean IDF (k_q=6, k_d=32)", topk_idf(6, True, True), topk_idf(32, True, True))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ColBERTEncoder] Loaded linear.weight -> dim=128 on cuda
Query 0: is lime good for diabetics
Gold passage length (tokens): 67
Naive IDF TOP-8 kept: [('honey', 6.91), ('moderately', 6.91), ('boiled', 6.21), ('pure', 6.21), ('meat', 6.21), ('meals', 5.81), ('fish', 5.52), ('juice', 5.3)]
Naive IDF BOT-8 dropped: [('.', 0.04), ('.', 0.04), ('.', 0.04), ('.', 0.04), (',', 0.18), ('the', 0.2), ('of', 0.28), ('a', 0.5)]
---------------------------------------------------------------------------
Baseline (Full ColBERTv2)            | R@10= 99.0% | MRR@10= 60.6% | avg_q= 6.6 | avg_d= 69.2 | pairs= 457.7 | 0.306 ms
Naive IDF (k_q=8, k_d=8)             | R@10= 71.0% | MRR@10= 36.3% | avg_q= 5.9 | avg_d=  8.0 | pairs=  47.4 | 0.104 ms
Clean+Dedup IDF (k_q=8, k_d=8)       | R@10= 67.0% | MRR@10= 33.2% | avg_q= 5.9 | avg_d=  8.0 | pairs=  47.4 | 0.105 ms
Clean+Dedup TF-IDF (k_q=8, k_d=8)    | R@10= 87.0% | MRR@10= 49.8% | avg_q= 5.9 | avg_d=  8.0 | pairs=  47.4 | 0.105 ms
Clean+Dedup TF-IDF (k_q=8,

---
## Section 3: Pipeline Smoke Test
Verifies 128-d ColBERTv2 projection loading, qualitative IDF token filtering, and end-to-end retrieval metrics on MS MARCO and BEIR TREC-COVID.

In [13]:
import math, re, time
import torch
from datasets import load_dataset
from src.data import build_msmarco_subset
from src.encoder import ColBERTEncoder
from src.idf_scorer import IDFScorer
from src.sparse_maxsim import pad_variable_embeddings, full_maxsim, sparse_maxsim
from src.evaluate import compute_metrics, benchmark_fn

enc = ColBERTEncoder()

# 1. Fit TRUE corpus-wide IDF once offline on 50,000 MS MARCO passages (~3 sec)
t0 = time.perf_counter()
val_ds = load_dataset("microsoft/ms_marco", "v2.1", split="validation")
bg_passages = []
for row in val_ds:
    for p in row["passages"]["passage_text"]:
        if p.strip():
            bg_passages.append(p)
        if len(bg_passages) >= 50000:
            break
    if len(bg_passages) >= 50000:
        break

corpus_idf = IDFScorer(enc.tokenizer).fit(bg_passages)
print(f"Fitted corpus-wide IDF on {corpus_idf.num_docs:,} passages ({len(corpus_idf.df):,} tokens) in {time.perf_counter()-t0:.2f}s")

# 2. Evaluate on MS MARCO seed 0 (100 queries x 1,000 passages)
queries, corpus, golds = build_msmarco_subset(seed=0, num_eval_queries=100, num_passages=1000)
q_embs, q_ids = enc.encode_queries(queries)
d_embs, d_ids = enc.encode_docs(corpus)

# Re-check Query 0 with corpus-wide IDF!
g_idx = next(iter(golds[0]))
toks = enc.tokenizer.convert_ids_to_tokens(d_ids[g_idx].tolist())
scores = corpus_idf.score_ids(d_ids[g_idx].tolist())
top8 = sorted(range(len(toks)), key=lambda i: scores[i], reverse=True)[:8]
print("Query 0:", queries[0])
print("Corpus-wide IDF TOP-8:", [(toks[i], round(scores[i], 2)) for i in top8])
print("-" * 82)

vocab_is_alpha = {}
def is_clean(tid):
    if tid not in vocab_is_alpha:
        tok = enc.tokenizer.convert_ids_to_tokens(int(tid))
        vocab_is_alpha[tid] = bool(re.match(r"^[a-zA-Z0-9]{2,}$", tok))
    return vocab_is_alpha[tid]

def make_rule(k, scorer, use_tf=False, dedup=True, clean=True):
    def fn(ids):
        id_list = ids.tolist()
        counts = {}
        if use_tf:
            for tid in id_list:
                counts[tid] = counts.get(tid, 0) + 1
        vals, seen = [], set()
        for tid in id_list:
            base = scorer.idf(tid)
            s = (1.0 + math.log(counts[tid])) * base if use_tf else base
            if clean and not is_clean(tid):
                s *= 0.1
            if dedup and tid in seen:
                s = -1.0
            seen.add(tid)
            vals.append(s)
        take = min(k, len(id_list))
        return torch.topk(torch.tensor(vals, device=ids.device), take).indices
    return fn

def eval_pair(name, q_rule, d_rule):
    Q_sel, Q_mask, avg_q = pad_variable_embeddings([q_embs[i][q_rule(q_ids[i])] for i in range(len(q_embs))])
    D_sel, D_mask, avg_d = pad_variable_embeddings([d_embs[i][d_rule(d_ids[i])] for i in range(len(d_embs))])
    run = lambda: [sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask) for i in range(len(q_embs))]
    all_s = run()
    ms = benchmark_fn(run, repeats=3, warmup=1) / len(q_embs)
    r10 = sum(compute_metrics(s, golds[i])["recall@10"] for i, s in enumerate(all_s)) / len(q_embs)
    mrr = sum(compute_metrics(s, golds[i])["mrr@10"] for i, s in enumerate(all_s)) / len(q_embs)
    print(f"{name:<38} | R@10={r10*100:5.1f}% | MRR={mrr*100:5.1f}% | pairs={avg_q*avg_d:5.1f} | {ms:.3f} ms")

D_full, M_full, avg_df = pad_variable_embeddings(d_embs)
avg_qf = sum(e.shape[0] for e in q_embs) / len(q_embs)
base_run = lambda: [full_maxsim(q_embs[i], D_full, M_full) for i in range(len(q_embs))]
base_s = base_run()
base_ms = benchmark_fn(base_run, repeats=3, warmup=1) / len(q_embs)
print(f"{'Baseline (Full ColBERTv2)':<38} | R@10={sum(compute_metrics(s, golds[i])['recall@10'] for i,s in enumerate(base_s)):5.1f}% | MRR={sum(compute_metrics(s, golds[i])['mrr@10'] for i,s in enumerate(base_s)):5.1f}% | pairs={avg_qf*avg_df:5.1f} | {base_ms:.3f} ms")

# Compare Corpus-wide IDF vs TF-IDF across symmetric and asymmetric budgets
for k in [8, 16, 24, 32]:
    eval_pair(f"Corpus IDF (sym k={k})", make_rule(k, corpus_idf, use_tf=False), make_rule(k, corpus_idf, use_tf=False))
for k in [8, 16, 24, 32]:
    eval_pair(f"Corpus TF-IDF (sym k={k})", make_rule(k, corpus_idf, use_tf=True), make_rule(k, corpus_idf, use_tf=True))
for kd in [16, 24, 32]:
    eval_pair(f"Corpus TF-IDF (k_q=8, k_d={kd})", make_rule(8, corpus_idf, use_tf=True), make_rule(kd, corpus_idf, use_tf=True))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ColBERTEncoder] Loaded linear.weight -> dim=128 on cuda
Fitted corpus-wide IDF on 50,000 passages (25,688 tokens) in 8.05s
Query 0: is lime good for diabetics
Corpus-wide IDF TOP-8: [('moderately', 7.88), ('boiled', 7.26), ('lime', 7.26), ('##bet', 6.6), ('meals', 6.46), ('varieties', 6.46), ('fruits', 6.21), ('fruits', 6.21)]
----------------------------------------------------------------------------------
Baseline (Full ColBERTv2)              | R@10= 99.0% | MRR= 60.6% | pairs=457.7 | 0.308 ms
Corpus IDF (sym k=8)                   | R@10= 85.0% | MRR= 39.3% | pairs= 47.4 | 0.107 ms
Corpus IDF (sym k=16)                  | R@10= 94.0% | MRR= 52.1% | pairs=105.8 | 0.106 ms
Corpus IDF (sym k=24)                  | R@10= 97.0% | MRR= 52.3% | pairs=158.4 | 0.108 ms
Corpus IDF (sym k=32)                  | R@10= 98.0% | MRR= 54.4% | pairs=210.5 | 0.108 ms
Corpus TF-IDF (sym k=8)                | R@10= 92.0% | MRR= 48.8% | pairs= 47.4 | 0.106 ms
Corpus TF-IDF (sym k=16)               | 

In [14]:
import py_compile
from pathlib import Path

# 1. src/idf_scorer.py — Offline corpus-wide IDF + punctuation detection
Path("src/idf_scorer.py").write_text(r'''"""Corpus-wide IDF importance scorer for QuickBERT.

Computes IDF(t) = log(N / df(t)) once offline over a reference passage collection.
Also identifies pure punctuation tokens while preserving WordPiece subwords (e.g., ##bet).
"""
import math
from typing import Dict, Iterable, List


class IDFScorer:
    """Computes and caches corpus-level IDF weights over tokenizer vocabulary."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.df: Dict[int, int] = {}
        self.num_docs: int = 0
        self._idf_cache: Dict[int, float] = {}
        self._content_cache: Dict[int, bool] = {}

    def fit(self, passages: Iterable[str]) -> "IDFScorer":
        """Compute document frequencies df(t) over a passage collection."""
        self.df = {}
        self.num_docs = 0
        for text in passages:
            self.num_docs += 1
            token_ids = self.tokenizer.encode(
                text, add_special_tokens=False, truncation=True, max_length=512
            )
            for tid in set(token_ids):
                self.df[tid] = self.df.get(tid, 0) + 1
        self._idf_cache = {
            tid: math.log(self.num_docs / freq)
            for tid, freq in self.df.items()
        }
        return self

    def idf(self, token_id: int) -> float:
        """Return IDF(t) = log(N / df(t)); unseen rare tokens receive log(N + 1)."""
        if token_id in self._idf_cache:
            return self._idf_cache[token_id]
        return math.log(self.num_docs + 1)

    def is_content_token(self, token_id: int) -> bool:
        """Return True for alphanumeric words and WordPiece subwords; False for punctuation."""
        if token_id not in self._content_cache:
            tok = self.tokenizer.convert_ids_to_tokens(int(token_id))
            stripped = tok[2:] if tok.startswith("##") else tok
            self._content_cache[token_id] = any(ch.isalnum() for ch in stripped)
        return self._content_cache[token_id]

    def score_ids(self, token_ids: List[int]) -> List[float]:
        """Return corpus IDF scores for a list of token IDs."""
        return [self.idf(int(tid)) for tid in token_ids]

    def inspect_query(self, query: str, k: int = 8) -> Dict[str, object]:
        """Return token-level IDF scores and kept/pruned breakdown for qualitative analysis."""
        ids = self.tokenizer.encode(query, add_special_tokens=False)
        tokens = self.tokenizer.convert_ids_to_tokens(ids)
        scores = self.score_ids(ids)
        ranked = sorted(range(len(ids)), key=lambda i: scores[i], reverse=True)
        keep_set = set(ranked[: min(k, len(ids))])
        kept = [(tokens[i], round(scores[i], 3)) for i in range(len(ids)) if i in keep_set]
        pruned = [(tokens[i], round(scores[i], 3)) for i in range(len(ids)) if i not in keep_set]
        return {"query": query, "tokens": tokens, "scores": scores, "kept": kept, "pruned": pruned}
''')

# 2. src/token_selector.py — Pure IDF, Deduplicated TF-IDF, Dynamic-k, and Random-k
Path("src/token_selector.py").write_text(r'''"""Training-free token selection for QuickBERT.

Supports:
  - idf:     pure corpus IDF top-k pruning (original QuickBERT baseline)
  - tfidf:   deduplicated, punctuation-filtered (1 + log tf) * IDF top-k pruning
  - dynamic: TF-IDF pruning with per-query dynamic k_q = max(4, |Q| // 2)
  - random:  uniformly random k tokens (ablation baseline)
"""
import math
from typing import List, Tuple
import torch
from src.idf_scorer import IDFScorer


class TokenSelector:
    """Prunes query and document token embeddings prior to late interaction."""

    def __init__(
        self,
        idf_scorer: IDFScorer,
        k: int = 8,
        mode: str = "tfidf",
        seed: int = 42,
    ):
        assert mode in {"idf", "tfidf", "dynamic", "random"}, f"Unknown mode: {mode}"
        self.idf_scorer = idf_scorer
        self.k = k
        self.mode = mode
        self.seed = seed

    def _score_sequence(self, id_list: List[int]) -> List[float]:
        if self.mode == "idf":
            return [self.idf_scorer.idf(tid) for tid in id_list]

        # TF-IDF with punctuation downweighting and first-occurrence deduplication
        counts = {}
        for tid in id_list:
            counts[tid] = counts.get(tid, 0) + 1

        scores = []
        seen = set()
        for tid in id_list:
            base = self.idf_scorer.idf(tid)
            val = (1.0 + math.log(counts[tid])) * base
            if not self.idf_scorer.is_content_token(tid):
                val *= 0.1
            if tid in seen:
                val = -1.0  # avoid wasting slots on duplicate token occurrences
            seen.add(tid)
            scores.append(val)
        return scores

    def select_batch(
        self,
        emb_list: List[torch.Tensor],
        ids_list: List[torch.Tensor],
        is_query: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, float]:
        """Select pruned token tensors and validity masks for a list of encoded texts."""
        n = len(emb_list)
        dim = emb_list[0].shape[-1]
        device = emb_list[0].device
        dtype = emb_list[0].dtype

        budgets = []
        for ids in ids_list:
            seq_len = int(ids.numel())
            if self.mode == "dynamic" and is_query:
                budgets.append(min(max(4, seq_len // 2), seq_len))
            else:
                budgets.append(min(self.k, seq_len))

        max_kept = max(budgets) if budgets else 1
        out = torch.zeros((n, max_kept, dim), dtype=dtype, device=device)
        mask = torch.zeros((n, max_kept), dtype=torch.bool, device=device)

        gen = torch.Generator(device="cpu").manual_seed(self.seed) if self.mode == "random" else None
        total_kept = 0

        for i in range(n):
            take = budgets[i]
            id_list = ids_list[i].tolist()
            seq_len = len(id_list)
            total_kept += take

            if take == seq_len:
                idx = torch.arange(seq_len, device=device)
            elif self.mode == "random":
                idx = torch.randperm(seq_len, generator=gen)[:take].to(device)
            else:
                seq_scores = torch.tensor(
                    self._score_sequence(id_list), dtype=torch.float32, device=device
                )
                idx = torch.topk(seq_scores, take, largest=True).indices

            out[i, :take] = emb_list[i][idx]
            mask[i, :take] = True

        return out, mask, total_kept / max(n, 1)
''')

# 3. src/data.py — Add 50,000-passage MS MARCO background loader for true offline IDF
Path("src/data.py").write_text(r'''"""Dataset loaders for MS MARCO passage retrieval and BEIR TREC-COVID."""
from typing import Dict, List, Set, Tuple
import random
from datasets import load_dataset


_MSMARCO_CACHE = {}


def _get_msmarco_validation():
    if "val" not in _MSMARCO_CACHE:
        _MSMARCO_CACHE["val"] = load_dataset("microsoft/ms_marco", "v2.1", split="validation")
    return _MSMARCO_CACHE["val"]


def load_msmarco_background(num_passages: int = 50000) -> List[str]:
    """Load unique passages from MS MARCO validation to compute offline corpus-wide IDF."""
    ds = _get_msmarco_validation()
    passages: List[str] = []
    for row in ds:
        for txt in row["passages"]["passage_text"]:
            if txt.strip():
                passages.append(txt)
            if len(passages) >= num_passages:
                return passages
    return passages


def build_msmarco_subset(
    seed: int = 0,
    num_eval_queries: int = 100,
    num_passages: int = 1000,
) -> Tuple[List[str], List[str], List[Set[int]]]:
    """Build a reproducible MS MARCO mini-corpus with guaranteed official positive passages."""
    ds = _get_msmarco_validation().shuffle(seed=seed)

    queries: List[str] = []
    query_gold_texts: List[List[str]] = []
    hard_neg_texts: List[str] = []
    filler_texts: List[str] = []

    scan_limit = min(len(ds), max(num_eval_queries * 10, num_passages // 2 + 500))
    for i in range(scan_limit):
        row = ds[i]
        passages = row["passages"]["passage_text"]
        selected = row["passages"]["is_selected"]
        pos = [p for p, s in zip(passages, selected) if s == 1 and p.strip()]
        neg = [p for p, s in zip(passages, selected) if s == 0 and p.strip()]

        if len(queries) < num_eval_queries and pos:
            queries.append(row["query"])
            query_gold_texts.append(pos)
            hard_neg_texts.extend(neg)
        else:
            filler_texts.extend([p for p in passages if p.strip()])

        if len(queries) >= num_eval_queries and (len(hard_neg_texts) + len(filler_texts)) >= num_passages * 2:
            break

    corpus: List[str] = []
    text_to_idx: Dict[str, int] = {}

    def add_passage(txt: str) -> int:
        if txt not in text_to_idx:
            text_to_idx[txt] = len(corpus)
            corpus.append(txt)
        return text_to_idx[txt]

    gold_sets: List[Set[int]] = []
    for pos_list in query_gold_texts:
        gold_sets.append({add_passage(p) for p in pos_list})

    for txt in hard_neg_texts:
        if len(corpus) >= num_passages:
            break
        add_passage(txt)

    for txt in filler_texts:
        if len(corpus) >= num_passages:
            break
        add_passage(txt)

    return queries, corpus, gold_sets


def build_trec_covid_subset(
    num_passages: int = 5000,
    seed: int = 42,
) -> Tuple[List[str], List[str], List[Dict[int, int]]]:
    """Load all 50 official BEIR TREC-COVID queries with graded relevance judgments."""
    q_ds = load_dataset("BeIR/trec-covid", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/trec-covid-qrels", split="test")
    corpus_ds = load_dataset("BeIR/trec-covid", "corpus", split="corpus")

    qid_to_rel: Dict[str, Dict[str, int]] = {}
    needed_doc_ids: Set[str] = set()
    for row in qrels_ds:
        score = int(row["score"])
        if score > 0:
            qid = str(row["query-id"])
            did = str(row["corpus-id"])
            qid_to_rel.setdefault(qid, {})[did] = score
            needed_doc_ids.add(did)

    rng = random.Random(seed)
    all_corpus_rows = {}
    distractors = []
    for row in corpus_ds:
        did = str(row["_id"])
        txt = (row.get("title", "") + " " + row.get("text", "")).strip()
        if not txt:
            continue
        if did in needed_doc_ids:
            all_corpus_rows[did] = txt
        elif len(distractors) < num_passages:
            distractors.append((did, txt))

    rng.shuffle(distractors)
    selected_doc_ids: List[str] = []
    seen: Set[str] = set()
    if len(all_corpus_rows) <= num_passages:
        for did in all_corpus_rows:
            selected_doc_ids.append(did)
            seen.add(did)
    else:
        for qid, rel_map in qid_to_rel.items():
            ranked_dids = sorted(rel_map.keys(), key=lambda d: rel_map[d], reverse=True)
            for did in ranked_dids[: max(10, num_passages // (2 * len(qid_to_rel)))]:
                if did in all_corpus_rows and did not in seen:
                    selected_doc_ids.append(did)
                    seen.add(did)
        for did in all_corpus_rows:
            if len(selected_doc_ids) >= int(num_passages * 0.65):
                break
            if did not in seen:
                selected_doc_ids.append(did)
                seen.add(did)

    corpus: List[str] = []
    did_to_idx: Dict[str, int] = {}
    for did in selected_doc_ids:
        did_to_idx[did] = len(corpus)
        corpus.append(all_corpus_rows[did])

    for did, txt in distractors:
        if len(corpus) >= num_passages:
            break
        if did not in did_to_idx:
            did_to_idx[did] = len(corpus)
            corpus.append(txt)

    queries: List[str] = []
    qrels_list: List[Dict[int, int]] = []
    for row in q_ds:
        qid = str(row["_id"])
        rel_map = qid_to_rel.get(qid, {})
        mapped = {did_to_idx[d]: s for d, s in rel_map.items() if d in did_to_idx}
        if mapped:
            queries.append(row["text"])
            qrels_list.append(mapped)

    return queries, corpus, qrels_list
''')

# 4. scripts/run_experiments.py — Complete official suite with 50k corpus-wide IDF
Path("scripts/run_experiments.py").write_text(r'''"""Unified experiment runner for QuickBERT (MS MARCO, TREC-COVID, and H100 benchmarks).

Usage:
  python scripts/run_experiments.py --mode baseline
  python scripts/run_experiments.py --mode quickbert --k 16 --pruner tfidf
  python scripts/run_experiments.py --mode all
"""
import argparse
import json
import os
import sys
import time
import yaml
import pandas as pd
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.data import build_msmarco_subset, build_trec_covid_subset, load_msmarco_background
from src.encoder import ColBERTEncoder
from src.evaluate import benchmark_fn, compute_metrics
from src.idf_scorer import IDFScorer
from src.sparse_maxsim import full_maxsim, pad_variable_embeddings, sparse_maxsim
from src.token_selector import TokenSelector
from src.utils import get_hardware_info, get_logger, save_json, set_seed


def evaluate_config(
    q_embs, q_ids, d_embs, d_ids, golds, idf_scorer,
    mode: str, k: int = 8, repeats: int = 5, seed: int = 42,
):
    """Evaluate full ColBERTv2 baseline or QuickBERT pruning configuration."""
    n_queries = len(q_embs)
    if mode == "baseline":
        D_mat, D_mask, avg_d = pad_variable_embeddings(d_embs)
        avg_q = sum(e.shape[0] for e in q_embs) / n_queries

        def score_all():
            return [full_maxsim(q_embs[i], D_mat, D_mask) for i in range(n_queries)]

        all_scores = score_all()
        total_ms = benchmark_fn(score_all, repeats=repeats, warmup=2)
    else:
        selector = TokenSelector(idf_scorer, k=k, mode=mode, seed=seed)
        Q_sel, Q_mask, avg_q = selector.select_batch(q_embs, q_ids, is_query=True)
        D_sel, D_mask, avg_d = selector.select_batch(d_embs, d_ids, is_query=False)

        def score_all():
            return [sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask) for i in range(n_queries)]

        all_scores = score_all()
        total_ms = benchmark_fn(score_all, repeats=repeats, warmup=2)

    recalls, mrrs, ndcgs = [], [], []
    for i, s in enumerate(all_scores):
        m = compute_metrics(s, golds[i], k=10)
        recalls.append(m["recall@10"])
        mrrs.append(m["mrr@10"])
        ndcgs.append(m["ndcg@10"])

    return {
        "recall@10": sum(recalls) / n_queries,
        "mrr@10": sum(mrrs) / n_queries,
        "ndcg@10": sum(ndcgs) / n_queries,
        "ms_per_query": total_ms / n_queries,
        "avg_q_tokens": avg_q,
        "avg_d_tokens": avg_d,
        "avg_interactions": avg_q * avg_d,
    }


def run_msmarco_5sub(encoder: ColBERTEncoder, corpus_idf: IDFScorer, cfg: dict, log):
    """Experiment 1: 5 random MS MARCO subsets (100 queries x 1,000 passages)."""
    log.info("=== Experiment 1: MS MARCO 5-Subset Sweep (50k Offline Corpus IDF) ===")
    seeds = cfg["subset_seeds"]
    ks = cfg["ks"]
    repeats = cfg["timing_repeats"]

    rows = []
    qualitative_saved = False

    for seed in seeds:
        queries, corpus, golds = build_msmarco_subset(
            seed=seed,
            num_eval_queries=cfg["num_eval_queries"],
            num_passages=cfg["num_passages"],
        )
        q_embs, q_ids = encoder.encode_queries(queries)
        d_embs, d_ids = encoder.encode_docs(corpus)

        if not qualitative_saved:
            samples = [
                corpus_idf.inspect_query("what is the population of raleigh", k=2),
                corpus_idf.inspect_query("what is the weather in raleigh today", k=4),
                corpus_idf.inspect_query(queries[0], k=4),
                corpus_idf.inspect_query(queries[1], k=4),
            ]
            save_json({"samples": samples}, "results/qualitative_idf.json")
            qualitative_saved = True

        base = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode="baseline", repeats=repeats)
        rows.append({
            "seed": seed, "pruner": "baseline", "k": 0,
            "recall@10": base["recall@10"], "mrr@10": base["mrr@10"], "ndcg@10": base["ndcg@10"],
            "recall_drop": 0.0, "ms_per_query": base["ms_per_query"], "speedup": 1.0,
            "avg_interactions": base["avg_interactions"],
        })

        for pruner in ["idf", "tfidf"]:
            for k in ks:
                res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode=pruner, k=k, repeats=repeats)
                rows.append({
                    "seed": seed, "pruner": pruner, "k": k,
                    "recall@10": res["recall@10"], "mrr@10": res["mrr@10"], "ndcg@10": res["ndcg@10"],
                    "recall_drop": base["recall@10"] - res["recall@10"],
                    "ms_per_query": res["ms_per_query"], "speedup": base["ms_per_query"] / res["ms_per_query"],
                    "avg_interactions": res["avg_interactions"],
                })

        # Dynamic-k and Random-k ablations at k=8 and k=16
        for mode_name in ["dynamic", "random"]:
            for k in [8, 16]:
                res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode=mode_name, k=k, repeats=repeats, seed=seed)
                rows.append({
                    "seed": seed, "pruner": mode_name, "k": k,
                    "recall@10": res["recall@10"], "mrr@10": res["mrr@10"], "ndcg@10": res["ndcg@10"],
                    "recall_drop": base["recall@10"] - res["recall@10"],
                    "ms_per_query": res["ms_per_query"], "speedup": base["ms_per_query"] / res["ms_per_query"],
                    "avg_interactions": res["avg_interactions"],
                })

    df = pd.DataFrame(rows)
    df.to_csv("results/msmarco_5sub_raw.csv", index=False)

    summary = (
        df.groupby(["pruner", "k"], sort=False)
        .agg(
            recall_mean=("recall@10", "mean"),
            recall_std=("recall@10", "std"),
            recall_drop_mean=("recall_drop", "mean"),
            recall_drop_std=("recall_drop", "std"),
            mrr_mean=("mrr@10", "mean"),
            ndcg_mean=("ndcg@10", "mean"),
            ms_mean=("ms_per_query", "mean"),
            speedup_mean=("speedup", "mean"),
            speedup_std=("speedup", "std"),
            avg_interactions=("avg_interactions", "mean"),
        )
        .reset_index()
    )
    summary.to_csv("results/msmarco_5sub_summary.csv", index=False)
    log.info("\n" + summary.to_string(index=False))
    return summary


def run_large_scale(encoder: ColBERTEncoder, corpus_idf: IDFScorer, cfg: dict, log):
    """Experiment 2A: Large-scale MS MARCO evaluation (500 queries x 10,000 passages)."""
    n_q = cfg["large_scale_queries"]
    n_p = cfg["large_scale_passages"]
    log.info(f"=== Experiment 2A: Large-Scale MS MARCO ({n_q} queries x {n_p} passages) ===")

    queries, corpus, golds = build_msmarco_subset(seed=42, num_eval_queries=n_q, num_passages=n_p)
    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs, d_ids = encoder.encode_docs(corpus)

    base = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode="baseline", repeats=3)
    rows = [{"pruner": "baseline", "k": 0, **base, "recall_drop": 0.0, "speedup": 1.0}]

    for pruner in ["idf", "tfidf"]:
        for k in cfg["ks"]:
            res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode=pruner, k=k, repeats=3)
            rows.append({
                "pruner": pruner, "k": k, **res,
                "recall_drop": base["recall@10"] - res["recall@10"],
                "speedup": base["ms_per_query"] / res["ms_per_query"],
            })

    for mode_name in ["dynamic", "random"]:
        for k in [8, 16]:
            res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode=mode_name, k=k, repeats=3)
            rows.append({
                "pruner": mode_name, "k": k, **res,
                "recall_drop": base["recall@10"] - res["recall@10"],
                "speedup": base["ms_per_query"] / res["ms_per_query"],
            })

    df = pd.DataFrame(rows)
    df.to_csv("results/msmarco_large_scale.csv", index=False)
    log.info("\n" + df[["pruner", "k", "recall@10", "mrr@10", "ndcg@10", "ms_per_query", "speedup", "avg_interactions"]].to_string(index=False))
    return df


def run_trec_covid(encoder: ColBERTEncoder, msmarco_idf: IDFScorer, cfg: dict, log):
    """Experiment 2B: Out-of-domain BEIR TREC-COVID evaluation (in-domain vs cross-domain IDF)."""
    n_p = cfg["trec_passages"]
    log.info(f"=== Experiment 2B: BEIR TREC-COVID (50 queries x {n_p} passages) ===")

    queries, corpus, qrels = build_trec_covid_subset(num_passages=n_p, seed=42)
    in_domain_idf = IDFScorer(encoder.tokenizer).fit(corpus)
    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs, d_ids = encoder.encode_docs(corpus)

    base = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode="baseline", repeats=5)
    rows = [{"idf_source": "none", "pruner": "baseline", "k": 0, **base, "ndcg_drop": 0.0, "speedup": 1.0}]

    for pruner in ["idf", "tfidf"]:
        for k in [4, 8, 16, 32]:
            res = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode=pruner, k=k, repeats=5)
            rows.append({
                "idf_source": "in_domain", "pruner": pruner, "k": k, **res,
                "ndcg_drop": base["ndcg@10"] - res["ndcg@10"],
                "speedup": base["ms_per_query"] / res["ms_per_query"],
            })

    # Cross-domain transfer using MS MARCO IDF on biomedical TREC-COVID passages
    for k in [8, 16, 32]:
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, msmarco_idf, mode="tfidf", k=k, repeats=5)
        rows.append({
            "idf_source": "cross_domain_msmarco", "pruner": "tfidf", "k": k, **res,
            "ndcg_drop": base["ndcg@10"] - res["ndcg@10"],
            "speedup": base["ms_per_query"] / res["ms_per_query"],
        })

    for k in [8, 16]:
        rnd = evaluate_config(q_embs, q_ids, d_embs, d_ids, qrels, in_domain_idf, mode="random", k=k, repeats=5)
        rows.append({
            "idf_source": "random", "pruner": "random", "k": k, **rnd,
            "ndcg_drop": base["ndcg@10"] - rnd["ndcg@10"],
            "speedup": base["ms_per_query"] / rnd["ms_per_query"],
        })

    df = pd.DataFrame(rows)
    df.to_csv("results/trec_covid_results.csv", index=False)
    log.info("\n" + df[["idf_source", "pruner", "k", "recall@10", "ndcg@10", "mrr@10", "speedup", "avg_interactions"]].to_string(index=False))
    return df


def run_hardware_bench(encoder: ColBERTEncoder, corpus_idf: IDFScorer, cfg: dict, log):
    """Experiment 3: Fixed-hardware H100 MIG scaling & end-to-end latency breakdown."""
    log.info("=== Experiment 3: Fixed-Hardware H100 MIG Latency Benchmark ===")
    hw = get_hardware_info()
    save_json(hw, "results/hardware_env.json")

    queries, corpus_10k, _ = build_msmarco_subset(seed=99, num_eval_queries=100, num_passages=10000)
    q_embs, q_ids = encoder.encode_queries(queries)
    d_embs_all, d_ids_all = encoder.encode_docs(corpus_10k)

    enc_ms_per_q = benchmark_fn(lambda: encoder.encode_queries(queries), repeats=5, warmup=2) / len(queries)
    prune_ms_per_q = benchmark_fn(
        lambda: TokenSelector(corpus_idf, k=16, mode="tfidf").select_batch(q_embs, q_ids, is_query=True),
        repeats=5, warmup=2,
    ) / len(queries)

    rows = []
    for corpus_n in [1000, 5000, 10000]:
        d_embs = d_embs_all[:corpus_n]
        d_ids = d_ids_all[:corpus_n]
        D_full, M_full, _ = pad_variable_embeddings(d_embs)

        def base_score():
            for i in range(len(q_embs)):
                full_maxsim(q_embs[i], D_full, M_full)

        base_ms = benchmark_fn(base_score, repeats=5, warmup=2) / len(q_embs)
        rows.append({
            "corpus_size": corpus_n, "k": 0, "method": "baseline",
            "score_ms_per_q": base_ms, "encode_ms_per_q": enc_ms_per_q,
            "prune_ms_per_q": 0.0, "e2e_ms_per_q": base_ms + enc_ms_per_q,
            "score_speedup": 1.0, "e2e_speedup": 1.0,
        })

        for k in cfg["ks"]:
            sel = TokenSelector(corpus_idf, k=k, mode="tfidf")
            Q_sel, Q_mask, _ = sel.select_batch(q_embs, q_ids, is_query=True)
            D_sel, D_mask, _ = sel.select_batch(d_embs, d_ids, is_query=False)

            def qb_score():
                for i in range(len(q_embs)):
                    sparse_maxsim(Q_sel[i], Q_mask[i], D_sel, D_mask)

            qb_ms = benchmark_fn(qb_score, repeats=5, warmup=2) / len(q_embs)
            e2e_ms = qb_ms + enc_ms_per_q + prune_ms_per_q
            rows.append({
                "corpus_size": corpus_n, "k": k, "method": f"quickbert_k={k}",
                "score_ms_per_q": qb_ms, "encode_ms_per_q": enc_ms_per_q,
                "prune_ms_per_q": prune_ms_per_q, "e2e_ms_per_q": e2e_ms,
                "score_speedup": base_ms / qb_ms,
                "e2e_speedup": (base_ms + enc_ms_per_q) / e2e_ms,
            })

    df = pd.DataFrame(rows)
    df.to_csv("results/hardware_benchmark.csv", index=False)
    log.info(f"Hardware: {hw['gpu_name']} | Query encoding: {enc_ms_per_q:.3f} ms/q | Query pruning: {prune_ms_per_q:.3f} ms/q")
    log.info("\n" + df.to_string(index=False))
    return df


def main():
    parser = argparse.ArgumentParser(description="Run QuickBERT retrieval & latency experiments.")
    parser.add_argument(
        "--mode",
        choices=["baseline", "quickbert", "msmarco_5sub", "large_scale", "trec_covid", "hardware_bench", "all"],
        default="all",
    )
    parser.add_argument("--config", type=str, default="configs/default.yaml")
    parser.add_argument("--k", type=int, default=None)
    parser.add_argument("--pruner", choices=["idf", "tfidf", "dynamic", "random"], default="tfidf")
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    set_seed(42)
    log = get_logger()
    hw = get_hardware_info()
    log.info(f"Environment: {json.dumps(hw)}")

    encoder = ColBERTEncoder(
        model_name=cfg["encoder"],
        q_maxlen=cfg["q_maxlen"],
        d_maxlen=cfg["d_maxlen"],
        batch_size=cfg["batch_size"],
    )

    t0 = time.perf_counter()
    bg_passages = load_msmarco_background(num_passages=50000)
    corpus_idf = IDFScorer(encoder.tokenizer).fit(bg_passages)
    log.info(f"Offline corpus IDF fitted on {corpus_idf.num_docs:,} passages ({len(corpus_idf.df):,} vocab tokens) in {time.perf_counter()-t0:.2f}s")

    if args.mode in {"baseline", "quickbert"}:
        queries, corpus, golds = build_msmarco_subset(
            seed=0, num_eval_queries=cfg["num_eval_queries"], num_passages=cfg["num_passages"]
        )
        q_embs, q_ids = encoder.encode_queries(queries)
        d_embs, d_ids = encoder.encode_docs(corpus)
        k_val = args.k if args.k is not None else cfg["k"]
        run_mode = "baseline" if args.mode == "baseline" else args.pruner
        res = evaluate_config(q_embs, q_ids, d_embs, d_ids, golds, corpus_idf, mode=run_mode, k=k_val, repeats=cfg["timing_repeats"])
        log.info(f"[{args.mode.upper()} mode={run_mode} k={k_val if run_mode!='baseline' else 'full'}] {res}")
        return

    if args.mode in {"msmarco_5sub", "all"}:
        run_msmarco_5sub(encoder, corpus_idf, cfg, log)
    if args.mode in {"large_scale", "all"}:
        run_large_scale(encoder, corpus_idf, cfg, log)
    if args.mode in {"trec_covid", "all"}:
        run_trec_covid(encoder, corpus_idf, cfg, log)
    if args.mode in {"hardware_bench", "all"}:
        run_hardware_bench(encoder, corpus_idf, cfg, log)


if __name__ == "__main__":
    main()
''')

for fname in ["src/idf_scorer.py", "src/token_selector.py", "src/data.py", "scripts/run_experiments.py"]:
    py_compile.compile(fname, doraise=True)
print("All Python modules compiled cleanly with zero syntax errors!")

All Python modules compiled cleanly with zero syntax errors!


---
## Section 4: Diagnostic — Subset IDF vs. Offline 50k Corpus IDF & TF-IDF
Analyzes why naive subset IDF degrades passage recall and validates offline 50,000-passage corpus IDF + deduplicated TF-IDF pruning.

In [15]:
!bash scripts/run_ablation.sh

2026-11-24 05:37:37,535 | INFO | Environment: {"device": "cuda", "gpu_name": "NVIDIA H100 80GB HBM3 MIG 1g.10gb", "cuda_version": "12.8", "torch_version": "2.10.0+cu128", "python_version": "3.13.11", "cpu_count": 20}
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 3956.83it/s]
[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[ColBERTEncoder] Loaded linear.weight -> dim=128 on cuda
2026-11-24 05:37:48,025 | INFO | Offline corpus IDF fitted on 50,000 passages (25,688 vocab tokens) in 8.38s
2026-11-24 05:37:48,025 | INFO | === Experiment 1: MS MARCO 5-Subset Sweep (50k Offline Corpus IDF) ===
2026-11-24 05:38:09,371 | INFO | 
  pruner  k  recall_mean  recall_std  recall_drop_mean  recall_drop_std  mrr_mean  ndcg_mean  ms_mean  speedup_m

---
## Section 5: Official H100 Experiment Suite (Main + Post-Presentation Additions)
Executes all 4 reproducible benchmarks on `NVIDIA H100 80GB HBM3 MIG 1g.10gb` and writes CSV/JSON logs to `results/`:
1. **Main Evaluation:** MS MARCO 5-subset sweep (100 queries × 1,000 passages, 5 seeds)
2. **Committed Addition 1:** Large-scale MS MARCO (500 queries × 10,000 passages)
3. **Committed Addition 2:** Out-of-domain BEIR TREC-COVID transfer (50 queries × 5,000 passages)
4. **Committed Addition 3:** Fixed-hardware H100 latency scaling & end-to-end breakdown

In [16]:
import py_compile
from pathlib import Path

Path("scripts/make_figures.py").write_text(r'''"""Generates all 6 publication-grade figures for the QuickBERT NeurIPS 2024 report.

Reads directly from CSV/JSON outputs produced by scripts/run_experiments.py (no hardcoded results).
Outputs saved to results/figures/ at 300 DPI.
"""
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd

os.makedirs("results/figures", exist_ok=True)
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10.5,
    "legend.fontsize": 9,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "figure.dpi": 300,
})


def make_pipeline_figure():
    """Figure 1: Clean vector-quality QuickBERT architecture pipeline diagram."""
    fig, ax = plt.subplots(figsize=(7.6, 5.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8.2)
    ax.axis("off")

    def draw_box(x, y, w, h, text, fc, ec, bold=True, fontsize=9.5):
        box = patches.FancyBboxPatch(
            (x - w / 2, y - h / 2), w, h,
            boxstyle="round,pad=0.12,rounding_size=0.18",
            facecolor=fc, edgecolor=ec, linewidth=1.6
        )
        ax.add_patch(box)
        ax.text(
            x, y, text, ha="center", va="center",
            fontsize=fontsize, fontweight="bold" if bold else "normal", color="#1a202c"
        )

    def arrow(x1, y1, x2, y2):
        ax.annotate(
            "", xy=(x2, y2), xytext=(x1, y1),
            arrowprops=dict(arrowstyle="-|>,head_length=0.55,head_width=0.3", color="#2d3748", lw=1.5)
        )

    # Query branch (left) & Document branch (right)
    draw_box(2.3, 7.4, 3.3, 0.95, "Query Tokens Q\n(ColBERTv2 128-d Embeddings)", "#dbeafe", "#2563eb")
    draw_box(7.7, 7.4, 3.3, 0.95, "Document Tokens D\n(ColBERTv2 128-d Embeddings)", "#dbeafe", "#2563eb")

    draw_box(2.3, 5.7, 3.3, 0.95, "Offline Corpus IDF Lookup\n+ TF-IDF Deduplication", "#dcfce7", "#16a34a")
    draw_box(7.7, 5.7, 3.3, 0.95, "Offline Corpus IDF Lookup\n+ TF-IDF Deduplication", "#dcfce7", "#16a34a")

    draw_box(2.3, 4.0, 3.3, 0.95, "Select Top-k Query Tokens\n(Fixed k or Dynamic k)", "#fef3c7", "#d97706")
    draw_box(7.7, 4.0, 3.3, 0.95, "Select Top-k Doc Tokens\n(No Retraining / 0 Params)", "#fef3c7", "#d97706")

    draw_box(5.0, 2.25, 5.4, 1.05, "Sparse ColBERT MaxSim Interaction\nO(|Q| x |D|)  -->  O(k^2)  (Up to 97% Fewer Pairs)", "#ede9fe", "#7c3aed")
    draw_box(5.0, 0.75, 3.4, 0.85, "Relevance Score & Ranked Passages", "#fee2e2", "#dc2626")

    # Arrows
    arrow(2.3, 6.85, 2.3, 6.25)
    arrow(7.7, 6.85, 7.7, 6.25)
    arrow(2.3, 5.15, 2.3, 4.55)
    arrow(7.7, 5.15, 7.7, 4.55)

    # Merge lines into Sparse MaxSim
    ax.plot([2.3, 2.3, 3.6], [3.45, 2.95, 2.95], color="#2d3748", lw=1.5)
    arrow(3.6, 2.95, 3.6, 2.84)
    ax.plot([7.7, 7.7, 6.4], [3.45, 2.95, 2.95], color="#2d3748", lw=1.5)
    arrow(6.4, 2.95, 6.4, 2.84)

    arrow(5.0, 1.65, 5.0, 1.24)

    plt.tight_layout()
    plt.savefig("results/figures/fig1_pipeline.png", dpi=300, bbox_inches="tight")
    plt.close()


def make_tradeoff_figure():
    """Figure 2: MS MARCO 5-Subset & 10k Large-Scale Speed vs Accuracy Trade-off."""
    df5 = pd.read_csv("results/msmarco_5sub_summary.csv")
    df10k = pd.read_csv("results/msmarco_large_scale.csv")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))

    # Panel A: 5-Subset MS MARCO (Recall@10 with error bars: IDF vs TF-IDF vs Random)
    base5 = df5[df5["pruner"] == "baseline"].iloc[0]
    idf5 = df5[df5["pruner"] == "idf"].sort_values("k")
    tfidf5 = df5[df5["pruner"] == "tfidf"].sort_values("k")
    rnd5 = df5[df5["pruner"] == "random"].sort_values("k")

    ax1.axhline(base5["recall_mean"] * 100, color="#4a5568", ls="--", lw=1.4, label=f"Full ColBERTv2 ({base5['recall_mean']*100:.1f}%)")
    ax1.errorbar(tfidf5["k"], tfidf5["recall_mean"] * 100, yerr=tfidf5["recall_std"] * 100,
                 marker="o", color="#2563eb", lw=2, capsize=3.5, label="QuickBERT-TFIDF (Ours)")
    ax1.errorbar(idf5["k"], idf5["recall_mean"] * 100, yerr=idf5["recall_std"] * 100,
                 marker="^", color="#dc2626", lw=1.8, ls="-.", capsize=3.5, label="QuickBERT-IDF (Pure)")
    ax1.errorbar(rnd5["k"], rnd5["recall_mean"] * 100, yerr=rnd5["recall_std"] * 100,
                 marker="s", color="#d97706", lw=1.6, ls=":", capsize=3.5, label="Random-k Ablation")
    ax1.set_xscale("log", base=2)
    ax1.set_xticks([2, 4, 8, 16, 32])
    ax1.set_xticklabels(["2", "4", "8", "16", "32"])
    ax1.set_xlabel("Retained Tokens per Side (k)")
    ax1.set_ylabel("Recall@10 (%)")
    ax1.set_title("(a) MS MARCO 5-Subset Sweep (mean ± std)")
    ax1.set_ylim(40, 102)
    ax1.grid(True, alpha=0.25)
    ax1.legend(loc="lower right", frameon=True)

    # Panel B: Large-Scale 10k MS MARCO (Speedup vs Recall@10 dual-axis)
    base10 = df10k[df10k["pruner"] == "baseline"].iloc[0]
    tfidf10 = df10k[df10k["pruner"] == "tfidf"].sort_values("k")
    idf10 = df10k[df10k["pruner"] == "idf"].sort_values("k")

    l1 = ax2.plot(tfidf10["k"], tfidf10["speedup"], marker="o", color="#2563eb", lw=2, label="H100 Scoring Speedup (x)")
    ax2.set_xscale("log", base=2)
    ax2.set_xticks([2, 4, 8, 16, 32])
    ax2.set_xticklabels(["2", "4", "8", "16", "32"])
    ax2.set_xlabel("Retained Tokens per Side (k)")
    ax2.set_ylabel("Scoring Speedup (x)", color="#2563eb")
    ax2.tick_params(axis="y", labelcolor="#2563eb")
    ax2.set_ylim(0, 13.0)
    ax2.grid(True, alpha=0.25)

    ax2_r = ax2.twinx()
    l2 = ax2_r.plot(tfidf10["k"], tfidf10["recall@10"] * 100, marker="s", color="#16a34a", ls="--", lw=2, label="QuickBERT-TFIDF Recall@10")
    l3 = ax2_r.plot(idf10["k"], idf10["recall@10"] * 100, marker="^", color="#dc2626", ls=":", lw=1.8, label="QuickBERT-IDF Recall@10")
    l4 = ax2_r.axhline(base10["recall@10"] * 100, color="#4a5568", ls="-.", lw=1.2, label=f"Baseline ({base10['recall@10']*100:.1f}%)")
    ax2_r.set_ylabel("Recall@10 (%)", color="#16a34a")
    ax2_r.tick_params(axis="y", labelcolor="#16a34a")
    ax2_r.set_ylim(30, 103)
    ax2.set_title("(b) Large-Scale MS MARCO (500q x 10k Passages)")

    lines = l1 + l2 + l3 + [l4]
    labels = [ln.get_label() for ln in lines]
    ax2.legend(lines, labels, loc="lower left", frameon=True, fontsize=8.2)

    plt.tight_layout()
    plt.savefig("results/figures/fig2_tradeoff.png", dpi=300, bbox_inches="tight")
    plt.close()


def make_interaction_and_dynamic_figure():
    """Figure 3: Token Interaction Collapse (left) & Fixed vs Dynamic k (right)."""
    df5 = pd.read_csv("results/msmarco_5sub_summary.csv")
    df_trec = pd.read_csv("results/trec_covid_results.csv")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.2))

    # Left: Real measured interaction collapse across MS MARCO and TREC-COVID
    ms_base = df5[df5["pruner"] == "baseline"]["avg_interactions"].iloc[0]
    ms_k8 = df5[(df5["pruner"] == "tfidf") & (df5["k"] == 8)]["avg_interactions"].iloc[0]
    ms_k16 = df5[(df5["pruner"] == "tfidf") & (df5["k"] == 16)]["avg_interactions"].iloc[0]
    tr_base = df_trec[df_trec["pruner"] == "baseline"]["avg_interactions"].iloc[0]
    tr_k8 = df_trec[(df_trec["idf_source"] == "cross_domain_msmarco") & (df_trec["k"] == 8)]["avg_interactions"].iloc[0]

    labels = ["MS MARCO\nFull", "MS MARCO\nk=16", "MS MARCO\nk=8", "TREC-COVID\nFull", "TREC-COVID\nk=8"]
    vals = [ms_base, ms_k16, ms_k8, tr_base, tr_k8]
    colors = ["#cbd5e1", "#60a5fa", "#2563eb", "#cbd5e1", "#7c3aed"]
    bars = ax1.bar(labels, vals, color=colors, edgecolor="#334155", linewidth=0.8)
    ax1.set_yscale("log")
    ax1.set_ylabel("Avg Token Pairs per (Q, D) [Log Scale]")
    ax1.set_title("(a) Measured Token Interaction Collapse")
    ax1.set_ylim(20, 6000)
    ax1.grid(True, axis="y", alpha=0.25)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width() / 2, v * 1.15, f"{v:.0f}", ha="center", va="bottom", fontweight="bold", fontsize=9)

    # Right: Fixed vs Dynamic k vs Random k on MS MARCO 5-Subset (Recall@10 & MRR@10)
    f8 = df5[(df5["pruner"] == "tfidf") & (df5["k"] == 8)].iloc[0]
    d8 = df5[(df5["pruner"] == "dynamic") & (df5["k"] == 8)].iloc[0]
    f16 = df5[(df5["pruner"] == "tfidf") & (df5["k"] == 16)].iloc[0]
    d16 = df5[(df5["pruner"] == "dynamic") & (df5["k"] == 16)].iloc[0]
    r8 = df5[(df5["pruner"] == "random") & (df5["k"] == 8)].iloc[0]

    x = np.arange(5)
    names = ["Random\nk=8", "Fixed\nk=8", "Dynamic\nk=8", "Fixed\nk=16", "Dynamic\nk=16"]
    r10s = [r8["recall_mean"] * 100, f8["recall_mean"] * 100, d8["recall_mean"] * 100, f16["recall_mean"] * 100, d16["recall_mean"] * 100]
    r10_err = [r8["recall_std"] * 100, f8["recall_std"] * 100, d8["recall_std"] * 100, f16["recall_std"] * 100, d16["recall_std"] * 100]
    c2 = ["#94a3b8", "#2563eb", "#f97316", "#1d4ed8", "#ea580c"]

    b2 = ax2.bar(x, r10s, yerr=r10_err, capsize=4, color=c2, edgecolor="#334155", linewidth=0.8)
    ax2.set_xticks(x)
    ax2.set_xticklabels(names)
    ax2.set_ylabel("Recall@10 (%)")
    ax2.set_ylim(88, 101)
    ax2.set_title("(b) Fixed vs. Dynamic k vs. Random Pruning")
    ax2.grid(True, axis="y", alpha=0.25)
    for bar, v in zip(b2, r10s):
        ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.55, f"{v:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=8.8)

    plt.tight_layout()
    plt.savefig("results/figures/fig3_interactions_and_dynamic.png", dpi=300, bbox_inches="tight")
    plt.close()


def make_trec_covid_figure():
    """Figure 4: Out-of-Domain TREC-COVID Transfer (In-Domain vs Cross-Domain IDF)."""
    df = pd.read_csv("results/trec_covid_results.csv")
    base = df[df["pruner"] == "baseline"].iloc[0]
    in_idf8 = df[(df["idf_source"] == "in_domain") & (df["pruner"] == "idf") & (df["k"] == 8)].iloc[0]
    in_tf8 = df[(df["idf_source"] == "in_domain") & (df["pruner"] == "tfidf") & (df["k"] == 8)].iloc[0]
    cr_tf8 = df[(df["idf_source"] == "cross_domain_msmarco") & (df["k"] == 8)].iloc[0]
    cr_tf32 = df[(df["idf_source"] == "cross_domain_msmarco") & (df["k"] == 32)].iloc[0]
    rnd8 = df[(df["pruner"] == "random") & (df["k"] == 8)].iloc[0]

    labels = ["Full\nColBERTv2", "Cross-Dom\nTF-IDF (k=8)", "Cross-Dom\nTF-IDF (k=32)", "In-Domain\nTF-IDF (k=8)", "In-Domain\nIDF (k=8)", "Random\n(k=8)"]
    ndcgs = [x["ndcg@10"] * 100 for x in [base, cr_tf8, cr_tf32, in_tf8, in_idf8, rnd8]]
    mrrs = [x["mrr@10"] * 100 for x in [base, cr_tf8, cr_tf32, in_tf8, in_idf8, rnd8]]

    x = np.arange(len(labels))
    w = 0.36
    fig, ax = plt.subplots(figsize=(9.2, 4.2))
    b1 = ax.bar(x - w / 2, ndcgs, w, label="NDCG@10 (%)", color="#2563eb", edgecolor="#1e3a8a", linewidth=0.7)
    b2 = ax.bar(x + w / 2, mrrs, w, label="MRR@10 (%)", color="#10b981", edgecolor="#065f46", linewidth=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Score (%)")
    ax.set_ylim(0, 108)
    ax.set_title("BEIR TREC-COVID (50 Queries x 5,000 Biomedical Passages): Domain-Shift Study")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(loc="upper right")

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 1.2, f"{h:.1f}", ha="center", va="bottom", fontsize=8.2, fontweight="bold")

    plt.tight_layout()
    plt.savefig("results/figures/fig4_trec_covid_transfer.png", dpi=300, bbox_inches="tight")
    plt.close()


def make_hardware_benchmark_figure():
    """Figure 5: Fixed-Hardware NVIDIA H100 MIG Latency Scaling & Breakdown."""
    df = pd.read_csv("results/hardware_benchmark.csv")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.2))

    # Left: Scoring & End-to-End Speedup across corpus sizes (N=1k, 5k, 10k)
    sizes = [1000, 5000, 10000]
    for k, col, mk in [(8, "#2563eb", "o"), (16, "#16a34a", "s"), (32, "#7c3aed", "^")]:
        sub = df[df["k"] == k].sort_values("corpus_size")
        ax1.plot(sub["corpus_size"], sub["score_speedup"], marker=mk, color=col, lw=2, label=f"k={k} (Scoring)")
        ax1.plot(sub["corpus_size"], sub["e2e_speedup"], marker=mk, color=col, lw=1.6, ls="--", alpha=0.75, label=f"k={k} (End-to-End)")
    ax1.set_xticks(sizes)
    ax1.set_xticklabels(["1,000", "5,000", "10,000"])
    ax1.set_xlabel("Candidate Corpus Size (Passages)")
    ax1.set_ylabel("Speedup over Full ColBERTv2 (x)")
    ax1.set_title("(a) H100 MIG Speedup Scaling vs. Corpus Size")
    ax1.grid(True, alpha=0.25)
    ax1.legend(ncol=2, fontsize=8, loc="upper left")

    # Right: Stacked End-to-End Latency Breakdown at N=10,000 passages
    d10 = df[df["corpus_size"] == 10000].copy()
    order = ["baseline", "quickbert_k=32", "quickbert_k=16", "quickbert_k=8", "quickbert_k=4"]
    d10 = d10.set_index("method").loc[order].reset_index()
    x_lbls = ["Baseline\n(Full)", "QuickBERT\nk=32", "QuickBERT\nk=16", "QuickBERT\nk=8", "QuickBERT\nk=4"]
    x = np.arange(len(x_lbls))

    enc = d10["encode_ms_per_q"].values
    prn = d10["prune_ms_per_q"].values
    scr = d10["score_ms_per_q"].values

    ax2.bar(x, enc, label="Query BERT Encoding", color="#94a3b8")
    ax2.bar(x, prn, bottom=enc, label="IDF Token Pruning", color="#f59e0b")
    ax2.bar(x, scr, bottom=enc + prn, label="MaxSim Scoring", color="#2563eb")
    totals = enc + prn + scr
    for xi, tot in zip(x, totals):
        ax2.text(xi, tot + 0.08, f"{tot:.2f} ms", ha="center", va="bottom", fontweight="bold", fontsize=8.8)

    ax2.set_xticks(x)
    ax2.set_xticklabels(x_lbls)
    ax2.set_ylabel("End-to-End Latency (ms / query)")
    ax2.set_ylim(0, max(totals) * 1.18)
    ax2.set_title("(b) Latency Breakdown at N=10,000 (NVIDIA H100 MIG)")
    ax2.grid(True, axis="y", alpha=0.25)
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig("results/figures/fig5_h100_scaling.png", dpi=300, bbox_inches="tight")
    plt.close()


def make_qualitative_idf_figure():
    """Figure 6: Real Offline Corpus IDF Scores for Query Tokens."""
    with open("results/qualitative_idf.json") as f:
        data = json.load(f)
    sample = data["samples"][1]  # "what is the weather in raleigh today"
    tokens = sample["tokens"]
    scores = sample["scores"]
    kept_set = {t for t, _ in sample["kept"]}
    colors = ["#2563eb" if t in kept_set else "#ef4444" for t in tokens]

    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    bars = ax.bar(tokens, scores, color=colors, edgecolor="#1e293b", linewidth=0.7)
    thresh = min(s for _, s in sample["kept"]) - 0.15
    ax.axhline(thresh, color="#475569", ls="--", lw=1.3, label=f"Top-4 Selection Cutoff")
    for bar, s in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, s + 0.18, f"{s:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

    # Legend patches
    kept_patch = patches.Patch(color="#2563eb", label="Retained (High IDF)")
    pruned_patch = patches.Patch(color="#ef4444", label="Pruned (Filler / Low IDF)")
    ax.legend(handles=[kept_patch, pruned_patch], loc="upper left")
    ax.set_ylabel("Corpus IDF Score: log(N / df(t))")
    ax.set_title(f'Real Corpus-Wide IDF Scores: "{sample["query"]}"')
    ax.set_ylim(0, max(scores) * 1.18)
    ax.grid(True, axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig("results/figures/fig6_qualitative_idf.png", dpi=300, bbox_inches="tight")
    plt.close()


if __name__ == "__main__":
    make_pipeline_figure()
    make_tradeoff_figure()
    make_interaction_and_dynamic_figure()
    make_trec_covid_figure()
    make_hardware_benchmark_figure()
    make_qualitative_idf_figure()
    print("Generated all 6 publication-grade figures in results/figures/:")
    for fn in sorted(os.listdir("results/figures")):
        if fn.startswith("fig"):
            print("  -", fn)
''')

py_compile.compile("scripts/make_figures.py", doraise=True)
!python scripts/make_figures.py

Generated all 6 publication-grade figures in results/figures/:
  - fig1_pipeline.png
  - fig2_tradeoff.png
  - fig3_interactions_and_dynamic.png
  - fig4_trec_covid_transfer.png
  - fig5_h100_scaling.png
  - fig6_qualitative_idf.png
